In [1]:
# ============================================================
# FULL PREPROCESSING PIPELINE — EDF → PKL
# Perubahan dari versi lama:
#    - Bandpass: 8.0-50.0 Hz
#    - Output: X_y_new_8_50hz.pkl (tidak overwrite pkl lama)
# Semua logic lain IDENTIK dengan pipeline sebelumnya
# ============================================================

# ============================================================
# CELL 1 — Import Libraries
# ============================================================
import os
import numpy as np
import pyedflib
import xml.etree.ElementTree as ET
import pickle
import neurokit2 as nk
from scipy.signal import resample_poly
from math import gcd

print("✅ Libraries imported")

# ============================================================
# CELL 2 — Configuration
# ============================================================
folder_path = r"E:\1D CNN\PSG and label"
SAVE_PATH   = r'E:\1D CNN\X_y_new_8_50hz.pkl'
TARGET_FS   = 200     # Hz — sama dengan pkl lama

# Segmentation
SEGMENT_SEC  = 60
MERGE_GAP    = 3
MIN_DURATION = 10
OVERLAP_THR  = 0.10

# AH event types
AH_EVENTS = [
    'Obstructive Apnea',
    'Mixed Apnea',
    'Central Apnea',
    'Obstructive Hypopnea',
    'Central Hypopnea',
    'SpO2 desaturation'
]
UNSURE_EVENTS = ['Unsure']

print(f"✅ Config set")
print(f"    Folder     : {folder_path}")
print(f"    Target fs  : {TARGET_FS} Hz")
print(f"    Segment    : {SEGMENT_SEC}s")
print(f"    Save to    : {SAVE_PATH}")

# ============================================================
# CELL 3 — Helper Functions
# ============================================================
def extract_number(name):
    """Sort subject IDs correctly: s1→1, s10→10"""
    return int(str(name)[1:].split('_')[0].split('.')[0])

def resample_ecg(signal, orig_fs, target_fs):
    """Resample ECG to target sampling rate"""
    if orig_fs == target_fs:
        return signal, target_fs
    g    = gcd(int(orig_fs), int(target_fs))
    up   = int(target_fs) // g
    down = int(orig_fs)   // g
    return resample_poly(signal, up, down), target_fs

def merge_and_filter_events(events):
    """
    Merge overlapping/nearby AH events.
    Rules:
    - Union area >= 10% → merge
    - Gap <= 3s → merge
    - Duration < 10s after merge → discard
    """
    ah_events = sorted(
        [e for e in events if e['type'] in AH_EVENTS],
        key=lambda x: x['start']
    )
    if not ah_events:
        return []

    merged  = []
    current = ah_events[0].copy()

    for e in ah_events[1:]:
        cur_start = current['start']
        cur_end   = current['start'] + current['duration']
        e_start   = e['start']
        e_end     = e['start'] + e['duration']

        intersection  = max(0, min(cur_end, e_end) - max(cur_start, e_start))
        union         = (cur_end - cur_start) + (e_end - e_start) - intersection
        overlap_ratio = intersection / union if union > 0 else 0
        gap           = e_start - cur_end

        if overlap_ratio >= OVERLAP_THR or gap <= MERGE_GAP:
            new_end             = max(cur_end, e_end)
            current['duration'] = new_end - cur_start
            current['type']     = 'Merged'
        else:
            merged.append(current)
            current = e.copy()

    merged.append(current)
    return [e for e in merged if e['duration'] >= MIN_DURATION]

print("✅ Helper functions defined")

# ============================================================
# CELL 4 — Load ECG from EDF Files
# ============================================================
if not os.path.isdir(folder_path):
    raise FileNotFoundError(f"Folder tidak ditemukan: {folder_path}")

edf_files = sorted(
    [f for f in os.listdir(folder_path) if f.lower().endswith('.edf')]
)
print(f"Found {len(edf_files)} EDF files")

all_ecg = {}

for filename in edf_files:
    filepath = os.path.join(folder_path, filename)
    subject  = filename.replace('.edf', '')
    try:
        f         = pyedflib.EdfReader(filepath)
        labels_ch = f.getSignalLabels()

        if 'ECG' in labels_ch:
            ecg_idx = labels_ch.index('ECG')
            fs      = f.getSampleFrequency(ecg_idx)
            ecg     = f.readSignal(ecg_idx)
            f._close()

            # Resample ke TARGET_FS kalau perlu
            ecg_resampled, new_fs = resample_ecg(ecg, fs, TARGET_FS)
            all_ecg[subject] = {'signal': ecg_resampled, 'fs': new_fs}
            print(f"✅ {subject} — {len(ecg_resampled)} samples | "
                  f"{new_fs} Hz | "
                  f"{len(ecg_resampled)/new_fs/60:.1f} min")
        else:
            f._close()
            print(f"⚠️  {subject} — No ECG channel found")

    except Exception as e:
        print(f"❌ {subject} — {e}")

print(f"\n✅ Loaded: {len(all_ecg)} subjects")

# ============================================================
# CELL 5 — Bandpass Filter
# PERUBAHAN: Bandpass diubah menjadi 8.0 - 50.0 Hz
# ============================================================
all_ecg_clean = {}

for subject in sorted(all_ecg.keys(), key=extract_number):
    ecg = all_ecg[subject]['signal']
    fs  = all_ecg[subject]['fs']

    # ⚠️ PERUBAHAN: lowcut=8.0, highcut=50.0
    ecg_filtered = nk.signal_filter(
        ecg,
        sampling_rate=fs,
        lowcut=8.0,    
        highcut=50.0,   
        method='butterworth'
    )

    all_ecg_clean[subject] = {
        'signal': ecg_filtered,
        'fs'    : fs
    }

    print(f"✅ {subject} — Bandpass 8.0–50.0 Hz applied | "
          f"range [{ecg_filtered.min():.3f}, {ecg_filtered.max():.3f}]")

print(f"\n✅ All {len(all_ecg_clean)} subjects filtered (8.0–50.0 Hz)")

# ============================================================
# CELL 6 — Segmentation 60s + Labeling from XML
# TIDAK ADA PERUBAHAN dari pipeline sebelumnya
# ============================================================
X                 = []
y                 = []
subject_ids       = []
all_merged_events = {}

print("=" * 60)
print("SEGMENTATION + LABELING (merge+filter applied)")
print(f"Bandpass: 8.0–50.0 Hz")
print("=" * 60)

for subject in sorted(all_ecg_clean.keys(), key=extract_number):
    ecg     = all_ecg_clean[subject]['signal']
    fs      = all_ecg_clean[subject]['fs']
    seg_len = int(SEGMENT_SEC * fs)   # 60 × 200 = 12000
    n_segs  = len(ecg) // seg_len

    # Read XML labels
    xml_path = os.path.join(folder_path, f"{subject}_re.XML")
    if not os.path.exists(xml_path):
        print(f"⚠️  {subject} — XML not found, skipping")
        continue

    tree   = ET.parse(xml_path)
    root   = tree.getroot()
    events = []
    for event in root.iter('ScoredEvent'):
        name     = event.find('Name')
        start    = event.find('Start')
        duration = event.find('Duration')
        if name is not None and start is not None:
            events.append({
                'type'    : name.text.strip(),
                'start'   : float(start.text),
                'duration': float(duration.text) if duration is not None else 0
            })

    # Merge + filter AH events
    merged_events             = merge_and_filter_events(events)
    all_merged_events[subject]= merged_events

    n_normal = n_ah = n_skipped = 0

    for i in range(n_segs):
        seg_start_sec = i * SEGMENT_SEC
        seg_end_sec   = seg_start_sec + SEGMENT_SEC
        sample_start  = i * seg_len
        sample_end    = sample_start + seg_len

        # Skip unsure segments
        is_unsure = any(
            e['type'] in UNSURE_EVENTS and
            e['start'] < seg_end_sec and
            (e['start'] + e['duration']) > seg_start_sec
            for e in events
        )
        if is_unsure:
            n_skipped += 1
            continue

        # Label dari merged events
        is_ah = any(
            e['start'] < seg_end_sec and
            (e['start'] + e['duration']) > seg_start_sec
            for e in merged_events
        )
        label = 1 if is_ah else 0

        # Extract segment + Z-score per segment
        segment  = ecg[sample_start:sample_end]
        seg_mean = np.mean(segment)
        seg_std  = np.std(segment)
        segment  = (segment - seg_mean) / (seg_std + 1e-8)

        X.append(segment)
        y.append(label)
        subject_ids.append(subject)

        if label == 1:
            n_ah += 1
        else:
            n_normal += 1

    print(f"✅ {subject:<6} | "
          f"Normal: {n_normal:>4} | "
          f"AH: {n_ah:>4} | "
          f"Skipped: {n_skipped:>3}")

# Convert to numpy
X           = np.array(X).reshape(-1, seg_len, 1)
y           = np.array(y)
subject_ids = np.array(subject_ids)

# ============================================================
# CELL 7 — Save pkl
# ============================================================
with open(SAVE_PATH, 'wb') as f:
    pickle.dump({
        'X'          : X,
        'y'          : y,
        'subject_ids': subject_ids
    }, f)

print(f"\n{'=' * 60}")
print(f"PREPROCESSING DONE — (bandpass 8.0–50.0 Hz)")
print(f"{'=' * 60}")
print(f"Total segments : {len(X)}")
print(f"Normal (0)     : {np.sum(y == 0)}")
print(f"AH     (1)     : {np.sum(y == 1)}")
print(f"Ratio          : {np.sum(y==0)/np.sum(y==1):.2f}x Normal > AH")
print(f"X shape        : {X.shape}")
print(f"Subjects        : {len(np.unique(subject_ids))}")
print(f"\nSaved → {SAVE_PATH}")
print(f"\nNext step:")
print(f"  Ganti path pkl di Exp 8:")
print(f"  with open(r'{SAVE_PATH}', 'rb') as f:")

✅ Libraries imported
✅ Config set
    Folder     : E:\1D CNN\PSG and label
    Target fs  : 200 Hz
    Segment    : 60s
    Save to    : E:\1D CNN\X_y_new_8_50hz.pkl
✅ Helper functions defined
Found 50 EDF files
✅ s1 — 4798600 samples | 200 Hz | 399.9 min
✅ s10 — 4753000 samples | 200 Hz | 396.1 min
✅ s11 — 4707000 samples | 200 Hz | 392.2 min
✅ s12 — 5032200 samples | 200 Hz | 419.4 min
✅ s13 — 4966600 samples | 200 Hz | 413.9 min
✅ s14 — 4717200 samples | 200 Hz | 393.1 min
✅ s15 — 4715800 samples | 200 Hz | 393.0 min
✅ s16 — 4779200 samples | 200 Hz | 398.3 min
✅ s17 — 4971800 samples | 200 Hz | 414.3 min
✅ s18 — 5074600 samples | 200 Hz | 422.9 min
✅ s19 — 4778000 samples | 200 Hz | 398.2 min
✅ s2 — 4493800 samples | 200 Hz | 374.5 min
✅ s20 — 5111200 samples | 200 Hz | 425.9 min
✅ s21 — 4734800 samples | 200 Hz | 394.6 min
✅ s22 — 4808000 samples | 200 Hz | 400.7 min
✅ s23 — 4957400 samples | 200 Hz | 413.1 min
✅ s24 — 4957600 samples | 200 Hz | 413.1 min
✅ s25 — 5041200 samples |

In [2]:
import numpy as np
import pickle
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from scipy.interpolate import interp1d
from biosppy.signals.ecg import hamilton_segmenter
from sklearn.metrics import (
    roc_auc_score, accuracy_score, confusion_matrix, roc_curve, f1_score
)
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CONFIGURATION
# ============================================================
FS         = 200
INTERP_FS  = 4
SEG_LEN    = 240
N_FOLDS    = 5
BATCH_SIZE = 32
EPOCHS     = 60
LR         = 1e-3
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SAVE_PATH  = r'E:\1D CNN\final_results_exp8_sw5fcv_X_y_new_8_50hz.pkl'

print(f"Device: {DEVICE}")
print("Exp 8 — Subject-Wise 5-Fold CV (Training + Validation)")

# ============================================================
# LOAD DATA
# ============================================================
with open(r'E:\1D CNN\X_y_new_8_50hz.pkl', 'rb') as f:
    data = pickle.load(f)

X_raw, y, subj = data['X'], np.array(data['y']), np.array(data['subject_ids'])

print(f"X shape  : {X_raw.shape}")
print(f"Normal   : {np.sum(y==0)}")
print(f"AH       : {np.sum(y==1)}")

# ============================================================
# SUBJECT-WISE NORMALIZATION
# ============================================================
def apply_subject_wise_scaling(X, subject_ids):
    X_scaled    = np.zeros_like(X)
    unique_subs = np.unique(subject_ids)
    for sub in unique_subs:
        mask     = (subject_ids == sub)
        sub_data = X[mask]
        mean     = np.mean(sub_data, axis=(0,2), keepdims=True)
        std      = np.std(sub_data,  axis=(0,2), keepdims=True) + 1e-8
        X_scaled[mask] = (sub_data - mean) / std
    return X_scaled

# ============================================================
# FEATURE EXTRACTION
# ============================================================
def extract_features(ecg_segment):
    ecg = np.asarray(ecg_segment).flatten()
    try:
        out     = hamilton_segmenter(signal=ecg, sampling_rate=FS)
        r_peaks = np.asarray(out[0], dtype=np.int64)
        if len(r_peaks) < 4: return None
        rr_ms    = np.diff(r_peaks) / FS * 1000.0
        rp_amp   = ecg[r_peaks[1:]]
        rr_times = r_peaks[1:] / FS
        t_uni    = np.arange(rr_times[0], rr_times[-1], 1/INTERP_FS)
        rr_uni   = interp1d(rr_times, rr_ms,   kind='cubic',
                            fill_value='extrapolate')(t_uni)
        rp_uni   = interp1d(rr_times, rp_amp,  kind='cubic',
                            fill_value='extrapolate')(t_uni)
        n = len(rr_uni)
        if n >= SEG_LEN:
            s      = (n - SEG_LEN) // 2
            rr_out = rr_uni[s:s+SEG_LEN]
            rp_out = rp_uni[s:s+SEG_LEN]
        else:
            pad    = SEG_LEN - n
            rr_out = np.pad(rr_uni, (pad//2, pad-pad//2), 'edge')
            rp_out = np.pad(rp_uni, (pad//2, pad-pad//2), 'edge')
        return np.stack([rr_out, rp_out], axis=0).astype(np.float32)
    except:
        return None

print("Extracting features...")
rr_list, y_list, subj_list = [], [], []
for i in range(len(X_raw)):
    feat = extract_features(X_raw[i])
    if feat is not None:
        rr_list.append(feat)
        y_list.append(y[i])
        subj_list.append(subj[i])

X_feat     = np.array(rr_list)
y_valid    = np.array(y_list)
subj_valid = np.array(subj_list)

# Subject-wise scaling
X_final = apply_subject_wise_scaling(X_feat, subj_valid)

# Sorted subject list — integer sort s1→s50
subjects = np.array(sorted(set(subj_valid),
           key=lambda x: int(str(x).replace('s',''))))

print(f"Valid    : {len(X_final)}")
print(f"Subjects : {len(subjects)}")

# ============================================================
# CREATE SUBJECT-WISE 5-FOLD SPLITS
# ============================================================
# Split 50 subjects into 5 groups of 10 subjects each
fold_size = len(subjects) // N_FOLDS  # 10 subjects per fold

subject_folds = []
for fold_idx in range(N_FOLDS):
    start_idx = fold_idx * fold_size
    end_idx = start_idx + fold_size
    fold_subjects = subjects[start_idx:end_idx]
    subject_folds.append(fold_subjects)
    print(f"Fold {fold_idx+1}: Subjects s{start_idx+1}-s{end_idx} "
          f"({len(fold_subjects)} subjects)")

print("\n" + "="*70)
print("Subject-Wise 5-Fold Cross-Validation Strategy:")
print("="*70)
for split_idx in range(N_FOLDS):
    val_fold = split_idx
    train_folds = [i for i in range(N_FOLDS) if i != val_fold]
    print(f"Split {split_idx+1}: Train=Folds {[f+1 for f in train_folds]} | "
          f"Val=Fold {val_fold+1}")
print("="*70 + "\n")

# ============================================================
# MODEL ARCHITECTURE
# ============================================================
class SEBlock(nn.Module):
    def __init__(self, channel, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channel, channel//reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channel//reduction, channel, bias=False),
            nn.Sigmoid())
    def forward(self, x):
        b, c, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1)
        return x * y.expand_as(x)

class MultiScaleBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        ch = out_ch // 4
        self.conv1_list = nn.ModuleList([
            nn.Conv1d(in_ch, ch, k, stride, k//2, bias=False)
            for k in [3,5,7,9]])
        self.bn1  = nn.BatchNorm1d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.conv2_list = nn.ModuleList([
            nn.Conv1d(out_ch, ch, k, 1, k//2, bias=False)
            for k in [3,5,7,9]])
        self.bn2      = nn.BatchNorm1d(out_ch)
        self.se       = SEBlock(out_ch)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_ch, out_ch, 1, stride, bias=False),
                nn.BatchNorm1d(out_ch))
    def forward(self, x):
        identity = self.shortcut(x)
        out = torch.cat([c(x)   for c in self.conv1_list], dim=1)
        out = self.relu(self.bn1(out))
        out = torch.cat([c(out) for c in self.conv2_list], dim=1)
        out = self.bn2(out)
        out = self.se(out)
        out += identity
        return self.relu(out)

class MultiScaleSEResNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(2, 64, 7, 3, 3, bias=False),
            nn.BatchNorm1d(64), nn.ReLU(True),
            nn.MaxPool1d(3, 2, 1))
        self.layer1 = self._make_layer(64,  64,  2, 1)
        self.layer2 = self._make_layer(64,  128, 2, 2)
        self.layer3 = self._make_layer(128, 256, 2, 2)
        self.layer4 = self._make_layer(256, 512, 2, 2)
        self.gap  = nn.AdaptiveAvgPool1d(1)
        self.fc   = nn.Linear(512, 1)
        self.drop = nn.Dropout(0.5)

    def _make_layer(self, in_c, out_c, blocks, stride):
        layers = [MultiScaleBlock(in_c, out_c, stride)]
        for _ in range(1, blocks):
            layers.append(MultiScaleBlock(out_c, out_c, 1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x); x = self.layer2(x)
        x = self.layer3(x); x = self.layer4(x)
        x = self.gap(x).squeeze(-1)
        return torch.sigmoid(self.fc(self.drop(x)))

# ============================================================
# SUBJECT-WISE 5-FOLD CV
# ============================================================
fold_results = []

for split_idx in range(N_FOLDS):
    print(f"\n{'='*70}")
    print(f"SPLIT {split_idx+1}/{N_FOLDS}")
    print(f"{'='*70}")
    
    # Validation fold = current split
    val_fold_idx = split_idx
    val_subjects = subject_folds[val_fold_idx]
    
    # Training folds = all other folds
    train_subjects = []
    for fold_idx in range(N_FOLDS):
        if fold_idx != val_fold_idx:
            train_subjects.extend(subject_folds[fold_idx])
    
    train_subjects = np.array(train_subjects)
    
    print(f"Training subjects: {len(train_subjects)} subjects")
    print(f"Validation subjects: {len(val_subjects)} subjects")
    
    # Create masks for training and validation
    train_mask = np.isin(subj_valid, train_subjects)
    val_mask = np.isin(subj_valid, val_subjects)
    
    X_train = X_final[train_mask]
    y_train = y_valid[train_mask]
    X_val   = X_final[val_mask]
    y_val   = y_valid[val_mask]
    
    print(f"Train: N={np.sum(y_train==0)}, AH={np.sum(y_train==1)} "
          f"(Total={len(y_train)})")
    print(f"Val:   N={np.sum(y_val==0)}, AH={np.sum(y_val==1)} "
          f"(Total={len(y_val)})")
    
    # Balanced sampler for training
    cls_count   = np.array([len(np.where(y_train==t)[0])
                             for t in np.unique(y_train)])
    weight      = 1. / cls_count
    samp_weight = np.array([weight[int(t)] for t in y_train])
    sampler     = WeightedRandomSampler(
                      torch.from_numpy(samp_weight),
                      len(samp_weight))
    
    train_loader = DataLoader(
        TensorDataset(torch.Tensor(X_train),
                      torch.Tensor(y_train)),
        batch_size=BATCH_SIZE, sampler=sampler)
    
    val_loader = DataLoader(
        TensorDataset(torch.Tensor(X_val),
                      torch.Tensor(y_val)),
        batch_size=BATCH_SIZE)
    
    model     = MultiScaleSEResNet().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                    optimizer, 'min', factor=0.5, patience=3)
    criterion = nn.BCELoss()
    
    best_auc    = 0
    best_state  = None
    patience_counter = 0
    
    for epoch in range(EPOCHS):
        # Training
        model.train()
        t_loss = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            out  = model(xb).view(-1)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            t_loss += loss.item()
        
        # Validation
        model.eval()
        y_prob_v = []
        with torch.no_grad():
            for xb, _ in val_loader:
                y_prob_v.extend(
                    model(xb.to(DEVICE)).cpu().numpy())
        
        val_auc = roc_auc_score(y_val, y_prob_v)
        scheduler.step(t_loss / len(train_loader))
        
        if val_auc > best_auc:
            best_auc   = val_auc
            best_state = {k: v.cpu().clone()
                          for k,v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= 12:
                print(f"  Early stop at epoch {epoch+1}")
                break
    
    # Final evaluation with best model
    model.load_state_dict(best_state)
    model.eval()
    y_prob = []
    with torch.no_grad():
        for xb, _ in val_loader:
            y_prob.extend(
                model(xb.to(DEVICE)).cpu().numpy())
    
    # Temporal smoothing
    y_prob = np.convolve(
        np.array(y_prob).flatten(),
        np.ones(3)/3, mode='same')
    
    # Youden Index threshold
    fpr, tpr, thresholds = roc_curve(y_val, y_prob)
    optimal_t = thresholds[np.argmax(tpr - fpr)]
    y_pred    = (y_prob >= optimal_t).astype(int)
    
    # Metrics
    auc  = roc_auc_score(y_val, y_prob)
    acc  = accuracy_score(y_val, y_pred)
    f1   = f1_score(y_val, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()
    sens = tp / (tp + fn)
    spec = tn / (tn + fp)
    
    print(f"\nSplit {split_idx+1} Results:")
    print(f"  AUC  = {auc:.4f}")
    print(f"  Acc  = {acc:.4f}")
    print(f"  F1   = {f1:.4f}")
    print(f"  Sens = {sens:.4f}")
    print(f"  Spec = {spec:.4f}")
    print(f"  Threshold = {optimal_t:.3f}")
    
    fold_results.append({
        'split': split_idx + 1,
        'val_fold': val_fold_idx + 1,
        'auc'    : auc,
        'acc'    : acc,
        'f1'     : f1,
        'sens'   : sens,
        'spec'   : spec
    })

# ============================================================
# FINAL SUMMARY
# ============================================================
print("\n" + "="*70)
print("FINAL SUMMARY — Subject-Wise 5-Fold CV")
print("="*70)

for m in ['auc', 'acc', 'f1', 'sens', 'spec']:
    vals = [r[m] for r in fold_results]
    print(f"Mean {m.upper():<5}: {np.mean(vals):.4f} ± {np.std(vals):.4f}")

print("\n" + "="*70)
print("Per-Split Results:")
print("="*70)
for r in fold_results:
    print(f"Split {r['split']}: AUC={r['auc']:.4f} | Acc={r['acc']:.4f} | "
          f"F1={r['f1']:.4f} | Sens={r['sens']:.4f} | Spec={r['spec']:.4f}")

with open(SAVE_PATH, 'wb') as f:
    pickle.dump(fold_results, f)
print(f"\nResults saved → {SAVE_PATH}")

Device: cuda
Exp 8 — Subject-Wise 5-Fold CV (Training + Validation)
X shape  : (20092, 12000, 1)
Normal   : 11548
AH       : 8544
Extracting features...
Valid    : 20091
Subjects : 50
Fold 1: Subjects s1-s10 (10 subjects)
Fold 2: Subjects s11-s20 (10 subjects)
Fold 3: Subjects s21-s30 (10 subjects)
Fold 4: Subjects s31-s40 (10 subjects)
Fold 5: Subjects s41-s50 (10 subjects)

Subject-Wise 5-Fold Cross-Validation Strategy:
Split 1: Train=Folds [2, 3, 4, 5] | Val=Fold 1
Split 2: Train=Folds [1, 3, 4, 5] | Val=Fold 2
Split 3: Train=Folds [1, 2, 4, 5] | Val=Fold 3
Split 4: Train=Folds [1, 2, 3, 5] | Val=Fold 4
Split 5: Train=Folds [1, 2, 3, 4] | Val=Fold 5


SPLIT 1/5
Training subjects: 40 subjects
Validation subjects: 10 subjects
Train: N=9891, AH=6152 (Total=16043)
Val:   N=1657, AH=2391 (Total=4048)
  Early stop at epoch 17

Split 1 Results:
  AUC  = 0.7865
  Acc  = 0.7058
  F1   = 0.7120
  Sens = 0.6156
  Spec = 0.8358
  Threshold = 0.565

SPLIT 2/5
Training subjects: 40 subjects
Valid

In [8]:
import numpy as np
import pickle
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from scipy.interpolate import interp1d
from biosppy.signals.ecg import hamilton_segmenter
from sklearn.metrics import (
    roc_auc_score, accuracy_score, confusion_matrix, roc_curve, f1_score
)
from sklearn.model_selection import StratifiedGroupKFold
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CONFIGURATION
# ============================================================
FS         = 200      # ✓ FIXED: Changed from 50 to 200 Hz
INTERP_FS  = 4
SEG_LEN    = 240
N_FOLDS    = 5
BATCH_SIZE = 32
EPOCHS     = 60
LR         = 1e-3
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_PATH  = r'E:\1D CNN\X_y_new_8_50hz.pkl'  # ✓ 8-50 Hz preprocessing
SAVE_PATH  = r'E:\1D CNN\final_results_exp8_sgkf_8_50hz.pkl'

print(f"Device: {DEVICE}")
print("Exp 8 — StratifiedGroupKFold with 8-50 Hz Preprocessing")

# ============================================================
# LOAD DATA
# ============================================================
with open(DATA_PATH, 'rb') as f:
    data = pickle.load(f)

X_raw = data['X']
y = np.array(data['y'])
subj = np.array(data['subject_ids'])

print(f"X shape  : {X_raw.shape}")
print(f"Normal   : {np.sum(y==0)}")
print(f"AH       : {np.sum(y==1)}")

# ============================================================
# SUBJECT-WISE NORMALIZATION
# ============================================================
def apply_subject_wise_scaling(X, subject_ids):
    X_scaled    = np.zeros_like(X)
    unique_subs = np.unique(subject_ids)
    for sub in unique_subs:
        mask     = (subject_ids == sub)
        sub_data = X[mask]
        mean     = np.mean(sub_data, axis=(0,2), keepdims=True)
        std      = np.std(sub_data,  axis=(0,2), keepdims=True) + 1e-8
        X_scaled[mask] = (sub_data - mean) / std
    return X_scaled

# ============================================================
# FEATURE EXTRACTION
# ============================================================
def extract_features(ecg_segment):
    ecg = np.asarray(ecg_segment).flatten()
    try:
        out     = hamilton_segmenter(signal=ecg, sampling_rate=FS)
        r_peaks = np.asarray(out[0], dtype=np.int64)
        if len(r_peaks) < 4: return None
        rr_ms    = np.diff(r_peaks) / FS * 1000.0
        rp_amp   = ecg[r_peaks[1:]]
        rr_times = r_peaks[1:] / FS
        t_uni    = np.arange(rr_times[0], rr_times[-1], 1/INTERP_FS)
        rr_uni   = interp1d(rr_times, rr_ms,   kind='cubic',
                            fill_value='extrapolate')(t_uni)
        rp_uni   = interp1d(rr_times, rp_amp,  kind='cubic',
                            fill_value='extrapolate')(t_uni)
        n = len(rr_uni)
        if n >= SEG_LEN:
            s      = (n - SEG_LEN) // 2
            rr_out = rr_uni[s:s+SEG_LEN]
            rp_out = rp_uni[s:s+SEG_LEN]
        else:
            pad    = SEG_LEN - n
            rr_out = np.pad(rr_uni, (pad//2, pad-pad//2), 'edge')
            rp_out = np.pad(rp_uni, (pad//2, pad-pad//2), 'edge')
        return np.stack([rr_out, rp_out], axis=0).astype(np.float32)
    except:
        return None

print("Extracting features...")
rr_list, y_list, subj_list = [], [], []
for i in range(len(X_raw)):
    feat = extract_features(X_raw[i])
    if feat is not None:
        rr_list.append(feat)
        y_list.append(y[i])
        subj_list.append(subj[i])

X_feat     = np.array(rr_list)
y_valid    = np.array(y_list)
subj_valid = np.array(subj_list)

print(f"Feature shape: {X_feat.shape}")

# Subject-wise scaling
print("Applying subject-wise scaling...")
X_final = apply_subject_wise_scaling(X_feat, subj_valid)

print(f"Final shape: {X_final.shape}")
print(f"Unique subjects: {len(np.unique(subj_valid))}")

# ============================================================
# MODEL ARCHITECTURE
# ============================================================
class SEBlock(nn.Module):
    def __init__(self, channel, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channel, channel//reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channel//reduction, channel, bias=False),
            nn.Sigmoid())
    def forward(self, x):
        b, c, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1)
        return x * y.expand_as(x)

class MultiScaleBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        ch = out_ch // 4
        self.conv1_list = nn.ModuleList([
            nn.Conv1d(in_ch, ch, k, stride, k//2, bias=False)
            for k in [3,5,7,9]])
        self.bn1  = nn.BatchNorm1d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.conv2_list = nn.ModuleList([
            nn.Conv1d(out_ch, ch, k, 1, k//2, bias=False)
            for k in [3,5,7,9]])
        self.bn2      = nn.BatchNorm1d(out_ch)
        self.se       = SEBlock(out_ch)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_ch, out_ch, 1, stride, bias=False),
                nn.BatchNorm1d(out_ch))
    def forward(self, x):
        identity = self.shortcut(x)
        out = torch.cat([c(x)   for c in self.conv1_list], dim=1)
        out = self.relu(self.bn1(out))
        out = torch.cat([c(out) for c in self.conv2_list], dim=1)
        out = self.bn2(out)
        out = self.se(out)
        out += identity
        return self.relu(out)

class MultiScaleSEResNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(2, 64, 7, 3, 3, bias=False),
            nn.BatchNorm1d(64), nn.ReLU(True),
            nn.MaxPool1d(3, 2, 1))
        self.layer1 = self._make_layer(64,  64,  2, 1)
        self.layer2 = self._make_layer(64,  128, 2, 2)
        self.layer3 = self._make_layer(128, 256, 2, 2)
        self.layer4 = self._make_layer(256, 512, 2, 2)
        self.gap  = nn.AdaptiveAvgPool1d(1)
        self.fc   = nn.Linear(512, 1)
        self.drop = nn.Dropout(0.5)

    def _make_layer(self, in_c, out_c, blocks, stride):
        layers = [MultiScaleBlock(in_c, out_c, stride)]
        for _ in range(1, blocks):
            layers.append(MultiScaleBlock(out_c, out_c, 1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x); x = self.layer2(x)
        x = self.layer3(x); x = self.layer4(x)
        x = self.gap(x).squeeze(-1)
        return torch.sigmoid(self.fc(self.drop(x)))

# ============================================================
# STRATIFIED GROUP K-FOLD CROSS-VALIDATION
# ============================================================
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

fold_results = []

print("\n" + "="*70)
print("Starting StratifiedGroupKFold Cross-Validation")
print("="*70)

for fold_idx, (train_idx, val_idx) in enumerate(
    sgkf.split(X_final, y_valid, groups=subj_valid)
):
    print(f"\n{'='*70}")
    print(f"FOLD {fold_idx+1}/{N_FOLDS}")
    print(f"{'='*70}")
    
    X_train = X_final[train_idx]
    y_train = y_valid[train_idx]
    X_val   = X_final[val_idx]
    y_val   = y_valid[val_idx]
    
    # Count subjects in this fold
    train_subjects = np.unique(subj_valid[train_idx])
    val_subjects = np.unique(subj_valid[val_idx])
    
    print(f"Training: {len(train_subjects)} subjects, "
          f"N={np.sum(y_train==0)}, AH={np.sum(y_train==1)} "
          f"(Total={len(y_train)})")
    print(f"Validation: {len(val_subjects)} subjects, "
          f"N={np.sum(y_val==0)}, AH={np.sum(y_val==1)} "
          f"(Total={len(y_val)})")
    
    # Balanced sampler for training
    cls_count   = np.array([len(np.where(y_train==t)[0])
                             for t in np.unique(y_train)])
    weight      = 1. / cls_count
    samp_weight = np.array([weight[int(t)] for t in y_train])
    sampler     = WeightedRandomSampler(
                      torch.from_numpy(samp_weight),
                      len(samp_weight))
    
    train_loader = DataLoader(
        TensorDataset(torch.Tensor(X_train),
                      torch.Tensor(y_train)),
        batch_size=BATCH_SIZE, sampler=sampler)
    
    val_loader = DataLoader(
        TensorDataset(torch.Tensor(X_val),
                      torch.Tensor(y_val)),
        batch_size=BATCH_SIZE)
    
    model     = MultiScaleSEResNet().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                    optimizer, 'min', factor=0.5, patience=3)
    criterion = nn.BCELoss()
    
    best_auc    = 0
    best_state  = None
    patience_counter = 0
    
    for epoch in range(EPOCHS):
        # Training
        model.train()
        t_loss = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            out  = model(xb).view(-1)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            t_loss += loss.item()
        
        # Validation
        model.eval()
        y_prob_v = []
        with torch.no_grad():
            for xb, _ in val_loader:
                y_prob_v.extend(
                    model(xb.to(DEVICE)).cpu().numpy())
        
        val_auc = roc_auc_score(y_val, y_prob_v)
        scheduler.step(t_loss / len(train_loader))
        
        if epoch % 10 == 0:
            print(f"  Epoch {epoch+1:03d} | "
                  f"Loss: {t_loss/len(train_loader):.4f} | "
                  f"AUC: {val_auc:.4f}")
        
        if val_auc > best_auc:
            best_auc   = val_auc
            best_state = {k: v.cpu().clone()
                          for k,v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= 12:
                print(f"  Early stop at epoch {epoch+1}")
                break
    
    # Final evaluation with best model
    model.load_state_dict(best_state)
    model.eval()
    y_prob = []
    with torch.no_grad():
        for xb, _ in val_loader:
            y_prob.extend(
                model(xb.to(DEVICE)).cpu().numpy())
    
    # Temporal smoothing
    y_prob = np.convolve(
        np.array(y_prob).flatten(),
        np.ones(3)/3, mode='same')
    
    # Youden Index threshold
    fpr, tpr, thresholds = roc_curve(y_val, y_prob)
    optimal_t = thresholds[np.argmax(tpr - fpr)]
    y_pred    = (y_prob >= optimal_t).astype(int)
    
    # Metrics
    auc  = roc_auc_score(y_val, y_prob)
    acc  = accuracy_score(y_val, y_pred)
    f1   = f1_score(y_val, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()
    sens = tp / (tp + fn)
    spec = tn / (tn + fp)
    
    print(f"\nFold {fold_idx+1} Results:")
    print(f"  AUC  = {auc:.4f}")
    print(f"  Acc  = {acc:.4f}")
    print(f"  F1   = {f1:.4f}")
    print(f"  Sens = {sens:.4f}")
    print(f"  Spec = {spec:.4f}")
    print(f"  Threshold = {optimal_t:.3f}")
    
    fold_results.append({
        'fold': fold_idx + 1,
        'n_train_subjects': len(train_subjects),
        'n_val_subjects': len(val_subjects),
        'auc'    : auc,
        'acc'    : acc,
        'f1'     : f1,
        'sens'   : sens,
        'spec'   : spec
    })

# ============================================================
# FINAL SUMMARY
# ============================================================
print("\n" + "="*70)
print("FINAL SUMMARY — StratifiedGroupKFold (8-50 Hz)")
print("="*70)

for m in ['auc', 'acc', 'f1', 'sens', 'spec']:
    vals = [r[m] for r in fold_results]
    print(f"Mean {m.upper():<5}: {np.mean(vals):.4f} ± {np.std(vals):.4f}")

print("\n" + "="*70)
print("Per-Fold Results:")
print("="*70)
for r in fold_results:
    print(f"Fold {r['fold']}: "
          f"Train={r['n_train_subjects']}subj, Val={r['n_val_subjects']}subj | "
          f"AUC={r['auc']:.4f} | Acc={r['acc']:.4f} | F1={r['f1']:.4f} | "
          f"Sens={r['sens']:.4f} | Spec={r['spec']:.4f}")

# Save results
results = {
    'fold_results': fold_results,
    'summary': {
        'mean_auc': np.mean([r['auc'] for r in fold_results]),
        'std_auc': np.std([r['auc'] for r in fold_results]),
        'mean_acc': np.mean([r['acc'] for r in fold_results]),
        'std_acc': np.std([r['acc'] for r in fold_results]),
        'mean_f1': np.mean([r['f1'] for r in fold_results]),
        'std_f1': np.std([r['f1'] for r in fold_results]),
        'mean_sens': np.mean([r['sens'] for r in fold_results]),
        'std_sens': np.std([r['sens'] for r in fold_results]),
        'mean_spec': np.mean([r['spec'] for r in fold_results]),
        'std_spec': np.std([r['spec'] for r in fold_results])
    }
}

with open(SAVE_PATH, 'wb') as f:
    pickle.dump(results, f)
print(f"\nResults saved → {SAVE_PATH}")


Device: cuda
Exp 8 — StratifiedGroupKFold with 8-50 Hz Preprocessing
X shape  : (20092, 12000, 1)
Normal   : 11548
AH       : 8544
Extracting features...
Feature shape: (20091, 2, 240)
Applying subject-wise scaling...
Final shape: (20091, 2, 240)
Unique subjects: 50

Starting StratifiedGroupKFold Cross-Validation

FOLD 1/5
Training: 40 subjects, N=8752, AH=7253 (Total=16005)
Validation: 10 subjects, N=2796, AH=1290 (Total=4086)
  Epoch 001 | Loss: 0.5738 | AUC: 0.7717
  Epoch 011 | Loss: 0.3379 | AUC: 0.7427
  Early stop at epoch 16

Fold 1 Results:
  AUC  = 0.8170
  Acc  = 0.7435
  F1   = 0.6433
  Sens = 0.7326
  Spec = 0.7486
  Threshold = 0.394

FOLD 2/5
Training: 40 subjects, N=9493, AH=6635 (Total=16128)
Validation: 10 subjects, N=2055, AH=1908 (Total=3963)
  Epoch 001 | Loss: 0.5810 | AUC: 0.7998
  Epoch 011 | Loss: 0.3668 | AUC: 0.7913
  Early stop at epoch 15

Fold 2 Results:
  AUC  = 0.8462
  Acc  = 0.7681
  F1   = 0.7321
  Sens = 0.6583
  Spec = 0.8701
  Threshold = 0.525

FO

In [11]:
import numpy as np
import pickle
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import (
    DataLoader,
    TensorDataset,
    WeightedRandomSampler
)

from scipy.interpolate import interp1d
from biosppy.signals.ecg import hamilton_segmenter

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    confusion_matrix,
    roc_curve
)

from sklearn.model_selection import StratifiedGroupKFold
from collections import Counter

import warnings

warnings.filterwarnings('ignore')

# ============================================================
# CONFIGURATION
# ============================================================

FS = 200
INTERP_FS = 4
SEG_LEN = 240  # 60s * 4Hz

N_FOLDS = 5
BATCH_SIZE = 32
EPOCHS = 60
LR = 1e-3

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

SAVE_PATH = r'E:\1D CNN\final_results_exp8_sw5fcv_X_y_new_8_50hz.pkl'

print(f"Device: {DEVICE}")
print("Starting Experiment 8: The Precision Framework (Subject-wise + SE-Block)")

# ============================================================
# 1. ENHANCED PREPROCESSING: SUBJECT-WISE NORMALIZATION
# ============================================================

def apply_subject_wise_scaling(X, subject_ids):
    """Menghitung Z-score secara terpisah untuk tiap subjek ID."""

    X_scaled = np.zeros_like(X)
    unique_subs = np.unique(subject_ids)

    for sub in unique_subs:
        mask = (subject_ids == sub)
        sub_data = X[mask]  # Shape: (n_segments, 2, 240)

        # Hitung mean dan std per channel (RR dan R-peak amplitude)
        # axis=(0,2) menghitung statistik di seluruh segmen dan waktu per channel
        mean = np.mean(sub_data, axis=(0, 2), keepdims=True)
        std = np.std(sub_data, axis=(0, 2), keepdims=True) + 1e-8

        X_scaled[mask] = (sub_data - mean) / std

    return X_scaled

# Load Data

with open(r'E:\1D CNN\X_y_new.pkl', 'rb') as f:
    data = pickle.load(f)

X_raw = data['X']
y = np.array(data['y'])
subj = np.array(data['subject_ids'])

def extract_features(ecg_segment):
    ecg = np.asarray(ecg_segment).flatten()

    try:
        out = hamilton_segmenter(signal=ecg, sampling_rate=FS)
        r_peaks = np.asarray(out[0], dtype=np.int64)

        if len(r_peaks) < 4:
            return None

        rr_ms = np.diff(r_peaks) / FS * 1000.0
        rp_amp = ecg[r_peaks[1:]]
        rr_times = r_peaks[1:] / FS

        t_uni = np.arange(rr_times[0], rr_times[-1], 1 / INTERP_FS)

        rr_uni = interp1d(
            rr_times,
            rr_ms,
            kind='cubic',
            fill_value='extrapolate'
        )(t_uni)

        rp_uni = interp1d(
            rr_times,
            rp_amp,
            kind='cubic',
            fill_value='extrapolate'
        )(t_uni)

        n = len(rr_uni)

        if n >= SEG_LEN:
            start = (n - SEG_LEN) // 2

            rr_out = rr_uni[start:start + SEG_LEN]
            rp_out = rp_uni[start:start + SEG_LEN]

        else:
            pad = SEG_LEN - n

            rr_out = np.pad(
                rr_uni,
                (pad // 2, pad - pad // 2),
                'edge'
            )

            rp_out = np.pad(
                rp_uni,
                (pad // 2, pad - pad // 2),
                'edge'
            )

        return np.stack([rr_out, rp_out], axis=0).astype(np.float32)

    except:
        return None

print("Extracting features and applying Subject-wise scaling...")

rr_list = []
y_list = []
subj_list = []

for i in range(len(X_raw)):
    feat = extract_features(X_raw[i])

    if feat is not None:
        rr_list.append(feat)
        y_list.append(y[i])
        subj_list.append(subj[i])

X_feat = np.array(rr_list)
y_valid = np.array(y_list)
subj_valid = np.array(subj_list)

# Eksekusi Normalisasi Per Subjek

X_final = apply_subject_wise_scaling(X_feat, subj_valid)

# ============================================================
# 2. ARCHITECTURE: MULTI-SCALE SE-RESNET
# ============================================================

class SEBlock(nn.Module):

    def __init__(self, channel, reduction=16):
        super().__init__()

        self.avg_pool = nn.AdaptiveAvgPool1d(1)

        self.fc = nn.Sequential(
            nn.Linear(channel, channel // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channel // reduction, channel, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.size()

        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1)

        return x * y.expand_as(x)

class MultiScaleBlock(nn.Module):

    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()

        ch = out_ch // 4

        self.conv1_list = nn.ModuleList([
            nn.Conv1d(in_ch, ch, k, stride, k // 2, bias=False)
            for k in [3, 5, 7, 9]
        ])

        self.bn1 = nn.BatchNorm1d(out_ch)
        self.relu = nn.ReLU(inplace=True)

        self.conv2_list = nn.ModuleList([
            nn.Conv1d(out_ch, ch, k, 1, k // 2, bias=False)
            for k in [3, 5, 7, 9]
        ])

        self.bn2 = nn.BatchNorm1d(out_ch)

        self.se = SEBlock(out_ch)

        self.shortcut = nn.Sequential()

        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_ch, out_ch, 1, stride, bias=False),
                nn.BatchNorm1d(out_ch)
            )

    def forward(self, x):
        identity = self.shortcut(x)

        out = torch.cat(
            [conv(x) for conv in self.conv1_list],
            dim=1
        )

        out = self.relu(self.bn1(out))

        out = torch.cat(
            [conv(out) for conv in self.conv2_list],
            dim=1
        )

        out = self.bn2(out)

        out = self.se(out)  # Channel Attention

        out += identity

        return self.relu(out)

class MultiScaleSEResNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv1d(2, 64, 7, 3, 3, bias=False),
            nn.BatchNorm1d(64),
            nn.ReLU(True),
            nn.MaxPool1d(3, 2, 1)
        )

        self.layer1 = self._make_layer(64, 64, 2, 1)
        self.layer2 = self._make_layer(64, 128, 2, 2)
        self.layer3 = self._make_layer(128, 256, 2, 2)
        self.layer4 = self._make_layer(256, 512, 2, 2)

        self.gap = nn.AdaptiveAvgPool1d(1)

        self.fc = nn.Linear(512, 1)

        self.drop = nn.Dropout(0.5)

    def _make_layer(self, in_c, out_c, blocks, stride):
        layers = [MultiScaleBlock(in_c, out_c, stride)]

        for _ in range(1, blocks):
            layers.append(MultiScaleBlock(out_c, out_c, 1))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.gap(x).squeeze(-1)

        return torch.sigmoid(self.fc(self.drop(x)))

# ============================================================
# 3. STRATIFIED CROSS-VALIDATION WITH BALANCED SAMPLER
# ============================================================

sgkf = StratifiedGroupKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=42
)

fold_results = []

for fold_idx, (train_idx, val_idx) in enumerate(
    sgkf.split(X_final, y_valid, groups=subj_valid)
):

    print(f"\n--- FOLD {fold_idx + 1} ---")

    X_train = X_final[train_idx]
    y_train = y_valid[train_idx]

    X_val = X_final[val_idx]
    y_val = y_valid[val_idx]

    # BALANCED SAMPLING

    class_sample_count = np.array([
        len(np.where(y_train == t)[0])
        for t in np.unique(y_train)
    ])

    weight = 1. / class_sample_count

    samples_weight = np.array([
        weight[int(t)]
        for t in y_train
    ])

    sampler = WeightedRandomSampler(
        torch.from_numpy(samples_weight),
        len(samples_weight)
    )

    train_loader = DataLoader(
        TensorDataset(
            torch.Tensor(X_train),
            torch.Tensor(y_train)
        ),
        batch_size=BATCH_SIZE,
        sampler=sampler
    )

    val_loader = DataLoader(
        TensorDataset(
            torch.Tensor(X_val),
            torch.Tensor(y_val)
        ),
        batch_size=BATCH_SIZE
    )

    model = MultiScaleSEResNet().to(DEVICE)

    optimizer = optim.Adam(model.parameters(), lr=LR)

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        'min',
        factor=0.5,
        patience=3
    )

    criterion = nn.BCELoss()

    best_auc = 0
    patience = 0

    best_model_state = None

    for epoch in range(EPOCHS):

        model.train()

        t_loss = 0

        for xb, yb in train_loader:

            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad()

            out = model(xb).view(-1)

            loss = criterion(out, yb)

            loss.backward()

            optimizer.step()

            t_loss += loss.item()

        model.eval()

        y_prob_v = []

        with torch.no_grad():

            for xb, _ in val_loader:
                y_prob_v.extend(
                    model(xb.to(DEVICE)).cpu().numpy()
                )

        val_auc = roc_auc_score(y_val, y_prob_v)

        scheduler.step(t_loss / len(train_loader))

        if val_auc > best_auc:

            best_auc = val_auc

            best_model_state = {
                k: v.cpu().clone()
                for k, v in model.state_dict().items()
            }

            patience = 0

        else:
            patience += 1

            if patience >= 12:
                break

    # FINAL EVALUATION FOR FOLD

    model.load_state_dict(best_model_state)

    model.eval()

    y_prob = []

    with torch.no_grad():

        for xb, _ in val_loader:
            y_prob.extend(
                model(xb.to(DEVICE)).cpu().numpy()
            )

    # Temporal Smoothing & Thresholding

    y_prob = np.convolve(
        np.array(y_prob).flatten(),
        np.ones(3) / 3,
        mode='same'
    )

    fpr, tpr, thresholds = roc_curve(y_val, y_prob)

    optimal_t = thresholds[np.argmax(tpr - fpr)]

    y_pred = (y_prob >= optimal_t).astype(int)

    auc = roc_auc_score(y_val, y_prob)

    acc = accuracy_score(y_val, y_pred)

    tn, fp, fn, tp = confusion_matrix(
        y_val,
        y_pred
    ).ravel()

    sens = tp / (tp + fn)

    spec = tn / (tn + fp)

    print(
        f"Result Fold {fold_idx+1}: "
        f"AUC={auc:.4f}, "
        f"Acc={acc:.4f}, "
        f"Sens={sens:.4f}, "
        f"Spec={spec:.4f}"
    )

    fold_results.append({
        'auc': auc,
        'acc': acc,
        'sens': sens,
        'spec': spec
    })

print("\n" + "=" * 50)

print("FINAL SUMMARY - EXPERIMENT 8 (THE PRECISION)")

print("=" * 50)

for m in ['auc', 'acc', 'sens', 'spec']:

    vals = [f[m] for f in fold_results]

    print(
        f"Mean {m.upper():<5}: "
        f"{np.mean(vals):.4f} ± {np.std(vals):.4f}"
    )

Device: cuda
Starting Experiment 8: The Precision Framework (Subject-wise + SE-Block)
Extracting features and applying Subject-wise scaling...

--- FOLD 1 ---
Result Fold 1: AUC=0.8035, Acc=0.7224, Sens=0.6885, Spec=0.7653

--- FOLD 2 ---
Result Fold 2: AUC=0.7524, Acc=0.6890, Sens=0.7388, Spec=0.6661

--- FOLD 3 ---
Result Fold 3: AUC=0.8649, Acc=0.7787, Sens=0.7191, Spec=0.8341

--- FOLD 4 ---
Result Fold 4: AUC=0.8016, Acc=0.7163, Sens=0.7140, Spec=0.7172

--- FOLD 5 ---
Result Fold 5: AUC=0.7961, Acc=0.7261, Sens=0.7390, Spec=0.7138

FINAL SUMMARY - EXPERIMENT 8 (THE PRECISION)
Mean AUC  : 0.8037 ± 0.0359
Mean ACC  : 0.7265 ± 0.0292
Mean SENS : 0.7199 ± 0.0187
Mean SPEC : 0.7393 ± 0.0569


In [12]:
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.interpolate import interp1d
from scipy.signal import welch
from scipy.stats import ttest_ind
from biosppy.signals.ecg import hamilton_segmenter
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CONFIGURATION
# ============================================================
FS = 200
INTERP_FS = 4
SEG_LEN = 240
DATA_PATH = r'E:\1D CNN\X_y_new.pkl'  # Using 0.8-10 Hz data
ANALYSIS_SAVE_PATH = r'E:\1D CNN\feature_analysis_report_0810hz.pkl'

print("="*70)
print("FEATURE ANALYSIS & TESTING FRAMEWORK (0.8-10 Hz Data)")
print("="*70)

# ============================================================
# LOAD DATA
# ============================================================
print("\n1. Loading data...")
with open(DATA_PATH, 'rb') as f:
    data = pickle.load(f)

X_raw = data['X']
y = np.array(data['y'])
subj = np.array(data['subject_ids'])

print(f"   X shape: {X_raw.shape}")
print(f"   Normal: {np.sum(y==0)}, AH: {np.sum(y==1)}")

# ============================================================
# BASELINE FEATURE EXTRACTION (Current 2 features)
# ============================================================
def extract_baseline_features(ecg_segment):
    """Current 2-channel feature extraction"""
    ecg = np.asarray(ecg_segment).flatten()
    try:
        out = hamilton_segmenter(signal=ecg, sampling_rate=FS)
        r_peaks = np.asarray(out[0], dtype=np.int64)
        if len(r_peaks) < 4:
            return None, None
        
        rr_ms = np.diff(r_peaks) / FS * 1000.0
        rp_amp = ecg[r_peaks[1:]]
        rr_times = r_peaks[1:] / FS
        
        t_uni = np.arange(rr_times[0], rr_times[-1], 1/INTERP_FS)
        rr_uni = interp1d(rr_times, rr_ms, kind='cubic', fill_value='extrapolate')(t_uni)
        rp_uni = interp1d(rr_times, rp_amp, kind='cubic', fill_value='extrapolate')(t_uni)
        
        n = len(rr_uni)
        if n >= SEG_LEN:
            s = (n - SEG_LEN) // 2
            rr_out = rr_uni[s:s+SEG_LEN]
            rp_out = rp_uni[s:s+SEG_LEN]
        else:
            pad = SEG_LEN - n
            rr_out = np.pad(rr_uni, (pad//2, pad-pad//2), 'edge')
            rp_out = np.pad(rp_uni, (pad//2, pad-pad//2), 'edge')
        
        # Return raw RR intervals for statistical analysis
        return np.stack([rr_out, rp_out], axis=0).astype(np.float32), rr_ms
        
    except:
        return None, None

# ============================================================
# ENHANCED FEATURE EXTRACTION (10 features)
# ============================================================
def extract_candidate_features(ecg_segment):
    """Extract ALL candidate features for analysis"""
    ecg = np.asarray(ecg_segment).flatten()
    
    try:
        out = hamilton_segmenter(signal=ecg, sampling_rate=FS)
        r_peaks = np.asarray(out[0], dtype=np.int64)
        
        if len(r_peaks) < 4:
            return None
        
        rr_ms = np.diff(r_peaks) / FS * 1000.0
        
        # Time-domain HRV features
        sdnn = np.std(rr_ms)
        rmssd = np.sqrt(np.mean(np.diff(rr_ms)**2))
        nn50 = np.sum(np.abs(np.diff(rr_ms)) > 50)
        pnn50 = (nn50 / len(rr_ms)) * 100 if len(rr_ms) > 0 else 0
        
        # Mean RR and HR
        mean_rr = np.mean(rr_ms)
        mean_hr = 60000 / mean_rr if mean_rr > 0 else 0
        
        # HRV Triangular Index
        hist, _ = np.histogram(rr_ms, bins=50)
        tri_index = len(rr_ms) / np.max(hist) if np.max(hist) > 0 else 0
        
        # Frequency-domain HRV features
        if len(rr_ms) >= 10:
            t_rr = np.cumsum(rr_ms) / 1000.0
            t_uniform = np.arange(0, t_rr[-1], 1.0)
            rr_interp = np.interp(t_uniform, t_rr, rr_ms)
            
            if len(rr_interp) >= 64:
                freqs, psd = welch(rr_interp, fs=1.0, nperseg=min(64, len(rr_interp)))
                
                # VLF power (0.003-0.04 Hz)
                vlf_mask = (freqs >= 0.003) & (freqs <= 0.04)
                vlf_power = np.trapz(psd[vlf_mask], freqs[vlf_mask]) if np.any(vlf_mask) else 0
                
                # LF power (0.04-0.15 Hz)
                lf_mask = (freqs >= 0.04) & (freqs <= 0.15)
                lf_power = np.trapz(psd[lf_mask], freqs[lf_mask]) if np.any(lf_mask) else 0
                
                # HF power (0.15-0.4 Hz)
                hf_mask = (freqs >= 0.15) & (freqs <= 0.4)
                hf_power = np.trapz(psd[hf_mask], freqs[hf_mask]) if np.any(hf_mask) else 0
                
                # Total power
                total_power = np.trapz(psd, freqs)
                
                # Ratios
                lf_hf_ratio = lf_power / hf_power if hf_power > 0 else 0
                lf_norm = lf_power / (lf_power + hf_power) if (lf_power + hf_power) > 0 else 0
                hf_norm = hf_power / (lf_power + hf_power) if (lf_power + hf_power) > 0 else 0
            else:
                vlf_power = lf_power = hf_power = total_power = 0
                lf_hf_ratio = lf_norm = hf_norm = 0
        else:
            vlf_power = lf_power = hf_power = total_power = 0
            lf_hf_ratio = lf_norm = hf_norm = 0
        
        # QRS-related features
        qrs_width_mean = 0
        qrs_amp_std = 0
        if len(r_peaks) >= 2:
            qrs_widths = []
            qrs_amps = []
            for i in range(len(r_peaks)):
                start = max(0, r_peaks[i] - int(0.08 * FS))
                end = min(len(ecg), r_peaks[i] + int(0.08 * FS))
                qrs_segment = ecg[start:end]
                qrs_widths.append(len(qrs_segment))
                qrs_amps.append(np.max(qrs_segment) - np.min(qrs_segment))
            qrs_width_mean = np.mean(qrs_widths)
            qrs_amp_std = np.std(qrs_amps)
        
        # Return dictionary of all features
        features = {
            # Basic (current)
            'mean_rr': mean_rr,
            'mean_hr': mean_hr,
            
            # Time-domain HRV
            'sdnn': sdnn,
            'rmssd': rmssd,
            'pnn50': pnn50,
            'tri_index': tri_index,
            
            # Frequency-domain HRV
            'vlf_power': vlf_power,
            'lf_power': lf_power,
            'hf_power': hf_power,
            'total_power': total_power,
            'lf_hf_ratio': lf_hf_ratio,
            'lf_norm': lf_norm,
            'hf_norm': hf_norm,
            
            # Morphological
            'qrs_width_mean': qrs_width_mean,
            'qrs_amp_std': qrs_amp_std
        }
        
        return features
        
    except:
        return None

# ============================================================
# ANALYZE SAMPLE SUBSET (faster testing)
# ============================================================
print("\n2. Extracting features from sample (5000 segments for analysis)...")
n_samples = min(5000, len(X_raw))
sample_indices = np.random.choice(len(X_raw), n_samples, replace=False)

baseline_features_list = []
candidate_features_list = []
labels_list = []

for idx in sample_indices:
    baseline_feat, _ = extract_baseline_features(X_raw[idx])
    candidate_feat = extract_candidate_features(X_raw[idx])
    
    if baseline_feat is not None and candidate_feat is not None:
        baseline_features_list.append(baseline_feat)
        candidate_features_list.append(candidate_feat)
        labels_list.append(y[idx])

labels = np.array(labels_list)
print(f"   Successfully extracted {len(labels)} samples")
print(f"   Normal: {np.sum(labels==0)}, AH: {np.sum(labels==1)}")

# ============================================================
# STATISTICAL ANALYSIS: COHEN'S D (Effect Size)
# ============================================================
print("\n3. Computing discriminative power (Cohen's d)...")
print("   Cohen's d interpretation:")
print("   - Small: 0.2-0.5")
print("   - Medium: 0.5-0.8")
print("   - Large: >0.8")
print()

def cohens_d(group1, group2):
    """Calculate Cohen's d effect size"""
    n1, n2 = len(group1), len(group2)
    var1, var2 = np.var(group1, ddof=1), np.var(group2, ddof=1)
    pooled_std = np.sqrt(((n1-1)*var1 + (n2-1)*var2) / (n1+n2-2))
    return (np.mean(group1) - np.mean(group2)) / pooled_std if pooled_std > 0 else 0

# Compute Cohen's d for all features
feature_analysis = {}

for feat_name in candidate_features_list[0].keys():
    feat_values = np.array([f[feat_name] for f in candidate_features_list])
    
    # Remove inf and nan
    valid_mask = np.isfinite(feat_values)
    feat_values_clean = feat_values[valid_mask]
    labels_clean = labels[valid_mask]
    
    normal_vals = feat_values_clean[labels_clean == 0]
    ah_vals = feat_values_clean[labels_clean == 1]
    
    if len(normal_vals) > 0 and len(ah_vals) > 0:
        # Cohen's d
        d = cohens_d(normal_vals, ah_vals)
        
        # T-test
        t_stat, p_val = ttest_ind(normal_vals, ah_vals)
        
        # Mean and std
        normal_mean, normal_std = np.mean(normal_vals), np.std(normal_vals)
        ah_mean, ah_std = np.mean(ah_vals), np.std(ah_vals)
        
        feature_analysis[feat_name] = {
            'cohens_d': abs(d),
            'p_value': p_val,
            'normal_mean': normal_mean,
            'normal_std': normal_std,
            'ah_mean': ah_mean,
            'ah_std': ah_std,
            'significant': p_val < 0.05
        }

# Sort by Cohen's d (discriminative power)
sorted_features = sorted(feature_analysis.items(), 
                        key=lambda x: x[1]['cohens_d'], 
                        reverse=True)

print("   FEATURE RANKING BY DISCRIMINATIVE POWER:")
print("   " + "="*68)
cohens_header = "Cohen's d"
print(f"   {'Feature':<20} {cohens_header:<12} {'p-value':<12} {'Significant':<12}")
print("   " + "="*68)

for feat_name, stats in sorted_features:
    sig_mark = "***" if stats['significant'] else ""
    print(f"   {feat_name:<20} {stats['cohens_d']:>8.4f}    "
          f"{stats['p_value']:>10.4e}   {sig_mark}")

print("   " + "="*68)

# ============================================================
# FEATURE SELECTION RECOMMENDATION
# ============================================================
print("\n4. FEATURE SELECTION RECOMMENDATION:")
print("   " + "="*68)

# Categorize features by Cohen's d
strong_features = [f for f, s in sorted_features if s['cohens_d'] > 0.5]
medium_features = [f for f, s in sorted_features if 0.2 <= s['cohens_d'] <= 0.5]
weak_features = [f for f, s in sorted_features if s['cohens_d'] < 0.2]

print(f"   STRONG (d > 0.5):   {len(strong_features)} features")
for feat in strong_features[:5]:  # Show top 5
    print(f"      • {feat:<20} (d={feature_analysis[feat]['cohens_d']:.3f})")
if len(strong_features) > 5:
    print(f"      ... and {len(strong_features)-5} more")

print()
print(f"   MEDIUM (0.2-0.5):   {len(medium_features)} features")
for feat in medium_features[:5]:
    print(f"      • {feat:<20} (d={feature_analysis[feat]['cohens_d']:.3f})")
if len(medium_features) > 5:
    print(f"      ... and {len(medium_features)-5} more")

print()
print(f"   WEAK (d < 0.2):     {len(weak_features)} features")
for feat in weak_features[:5]:
    print(f"      • {feat:<20} (d={feature_analysis[feat]['cohens_d']:.3f})")
if len(weak_features) > 5:
    print(f"      ... and {len(weak_features)-5} more")

# ============================================================
# RECOMMENDED FEATURE SETS
# ============================================================
print("\n5. RECOMMENDED FEATURE SETS:")
print("   " + "="*68)

# Option 1: Only strong features
recommended_strong = [f for f in strong_features 
                     if feature_analysis[f]['cohens_d'] > 0.5]
print(f"\n   OPTION 1: STRONG ONLY ({len(recommended_strong)} features)")
print(f"   Features: {', '.join(recommended_strong[:10])}")
if len(recommended_strong) > 10:
    print(f"            ... and {len(recommended_strong)-10} more")

# Option 2: Strong + Medium
recommended_strong_medium = strong_features + medium_features
print(f"\n   OPTION 2: STRONG + MEDIUM ({len(recommended_strong_medium)} features)")
print(f"   Features: {', '.join(recommended_strong_medium[:10])}")
if len(recommended_strong_medium) > 10:
    print(f"            ... and {len(recommended_strong_medium)-10} more")

# Option 3: Top N by Cohen's d
top_n = 10
recommended_top10 = [f for f, _ in sorted_features[:top_n]]
print(f"\n   OPTION 3: TOP {top_n} BY COHEN'S D")
for i, feat in enumerate(recommended_top10, 1):
    d_val = feature_analysis[feat]['cohens_d']
    print(f"      {i:2d}. {feat:<20} (d={d_val:.3f})")

# ============================================================
# CORRELATION ANALYSIS
# ============================================================
print("\n6. FEATURE CORRELATION ANALYSIS:")
print("   (Detecting redundant features)")
print("   " + "="*68)

# Build feature matrix
feature_names = list(candidate_features_list[0].keys())
feature_matrix = np.zeros((len(candidate_features_list), len(feature_names)))

for i, feat_dict in enumerate(candidate_features_list):
    for j, feat_name in enumerate(feature_names):
        feature_matrix[i, j] = feat_dict[feat_name]

# Replace inf/nan
feature_matrix = np.nan_to_num(feature_matrix, nan=0.0, posinf=0.0, neginf=0.0)

# Compute correlation matrix
corr_matrix = np.corrcoef(feature_matrix.T)

# Find highly correlated pairs (|r| > 0.9)
high_corr_pairs = []
for i in range(len(feature_names)):
    for j in range(i+1, len(feature_names)):
        if abs(corr_matrix[i, j]) > 0.9:
            high_corr_pairs.append((feature_names[i], feature_names[j], corr_matrix[i, j]))

if high_corr_pairs:
    print("   HIGH CORRELATION (|r| > 0.9) - Consider removing one:")
    for f1, f2, r in high_corr_pairs:
        print(f"      • {f1} ↔ {f2}: r={r:.3f}")
else:
    print("   ✓ No highly correlated features found (all |r| < 0.9)")

# ============================================================
# FINAL RECOMMENDATION
# ============================================================
print("\n" + "="*70)
print("7. FINAL RECOMMENDATION:")
print("="*70)

# Choose top features by Cohen's d, remove highly correlated
final_recommended = []
used_features = set()

for feat, _ in sorted_features:
    # Skip if highly correlated with already selected feature
    skip = False
    for selected in final_recommended:
        feat_idx = feature_names.index(feat)
        selected_idx = feature_names.index(selected)
        if abs(corr_matrix[feat_idx, selected_idx]) > 0.9:
            skip = True
            break
    
    if not skip:
        final_recommended.append(feat)
    
    # Stop at 10 features
    if len(final_recommended) >= 10:
        break

print(f"\nRECOMMENDED: {len(final_recommended)} FEATURES")
print("(Ranked by Cohen's d, redundant features removed)\n")

for i, feat in enumerate(final_recommended, 1):
    stats = feature_analysis[feat]
    print(f"{i:2d}. {feat:<20} | d={stats['cohens_d']:>6.3f} | "
          f"p={stats['p_value']:>10.4e} | "
          f"Normal={stats['normal_mean']:>8.2f}±{stats['normal_std']:>6.2f} | "
          f"AH={stats['ah_mean']:>8.2f}±{stats['ah_std']:>6.2f}")

# ============================================================
# SAVE ANALYSIS RESULTS
# ============================================================
analysis_results = {
    'feature_analysis': feature_analysis,
    'sorted_features': sorted_features,
    'recommended_features': final_recommended,
    'high_corr_pairs': high_corr_pairs,
    'strong_features': strong_features,
    'medium_features': medium_features,
    'weak_features': weak_features
}

with open(ANALYSIS_SAVE_PATH, 'wb') as f:
    pickle.dump(analysis_results, f)

print(f"\n✓ Analysis saved to: {ANALYSIS_SAVE_PATH}")

# ============================================================
# GENERATE IMPLEMENTATION CODE
# ============================================================
print("\n" + "="*70)
print("8. IMPLEMENTATION CODE SNIPPET:")
print("="*70)

print(f"""
Add these {len(final_recommended)} features to your extract_features() function:

def extract_enhanced_features(ecg_segment):
    ecg = np.asarray(ecg_segment).flatten()
    try:
        out = hamilton_segmenter(signal=ecg, sampling_rate=FS)
        r_peaks = np.asarray(out[0], dtype=np.int64)
        if len(r_peaks) < 4:
            return None
        
        rr_ms = np.diff(r_peaks) / FS * 1000.0
        
        # Extract recommended features:
""")

for feat in final_recommended:
    if feat in ['mean_rr', 'mean_hr']:
        print(f"        {feat} = ...")
    elif feat in ['sdnn', 'rmssd', 'pnn50', 'tri_index']:
        print(f"        {feat} = ...  # Time-domain HRV")
    elif 'power' in feat or 'lf_hf' in feat or 'norm' in feat:
        print(f"        {feat} = ...  # Frequency-domain HRV")
    elif 'qrs' in feat:
        print(f"        {feat} = ...  # Morphological")

print(f"""
        # Stack into ({len(final_recommended)}, 240) array
        # (Interpolate each feature to 240 samples)
        
        return features  # Shape: ({len(final_recommended)}, 240)
""")

print("\n" + "="*70)
print("ANALYSIS COMPLETE!")
print("="*70)
print("\nNext steps:")
print("1. Review the recommended features above")
print("2. Implement the enhanced feature extraction")
print("3. Update model input channels to", len(final_recommended))
print("4. Train and compare results")


FEATURE ANALYSIS & TESTING FRAMEWORK (0.8-10 Hz Data)

1. Loading data...
   X shape: (20092, 12000, 1)
   Normal: 11548, AH: 8544

2. Extracting features from sample (5000 segments for analysis)...
   Successfully extracted 5000 samples
   Normal: 2885, AH: 2115

3. Computing discriminative power (Cohen's d)...
   Cohen's d interpretation:
   - Small: 0.2-0.5
   - Medium: 0.5-0.8
   - Large: >0.8

   FEATURE RANKING BY DISCRIMINATIVE POWER:
   Feature              Cohen's d    p-value      Significant 
   mean_hr                0.4769    1.0977e-60   ***
   mean_rr                0.4673    2.1328e-58   ***
   qrs_amp_std            0.4490    3.7471e-54   ***
   tri_index              0.3012    1.2747e-25   ***
   sdnn                   0.2006    2.7483e-12   ***
   pnn50                  0.0657    2.1823e-02   ***
   qrs_width_mean         0.0367    1.9997e-01   
   rmssd                  0.0023    9.3606e-01   
   vlf_power              0.0000           nan   
   lf_power            

In [13]:
import numpy as np
import pickle
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from scipy.interpolate import interp1d
from biosppy.signals.ecg import hamilton_segmenter
from sklearn.metrics import (
    roc_auc_score, accuracy_score, confusion_matrix, roc_curve, f1_score
)
from sklearn.model_selection import StratifiedGroupKFold
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CONFIGURATION
# ============================================================
FS         = 200
INTERP_FS  = 4
SEG_LEN    = 240
N_FOLDS    = 5
BATCH_SIZE = 32
EPOCHS     = 60
LR         = 1e-3
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_PATH  = r'E:\1D CNN\X_y_new.pkl'  # 0.8-10 Hz data
SAVE_PATH  = r'E:\1D CNN\results_6feat_sgkf_0810hz.pkl'

print(f"Device: {DEVICE}")
print("Enhanced 6-Feature Model with StratifiedGroupKFold (0.8-10 Hz)")
print("="*70)
print("Testing if enhanced features help with old preprocessing")
print("="*70)

# ============================================================
# LOAD DATA
# ============================================================
with open(DATA_PATH, 'rb') as f:
    data = pickle.load(f)

X_raw = data['X']
y = np.array(data['y'])
subj = np.array(data['subject_ids'])

print(f"\nX shape: {X_raw.shape}")
print(f"Normal: {np.sum(y==0)}, AH: {np.sum(y==1)}")

# ============================================================
# SUBJECT-WISE NORMALIZATION
# ============================================================
def apply_subject_wise_scaling(X, subject_ids):
    X_scaled = np.zeros_like(X)
    unique_subs = np.unique(subject_ids)
    for sub in unique_subs:
        mask = (subject_ids == sub)
        sub_data = X[mask]
        mean = np.mean(sub_data, axis=(0,2), keepdims=True)
        std = np.std(sub_data, axis=(0,2), keepdims=True) + 1e-8
        X_scaled[mask] = (sub_data - mean) / std
    return X_scaled

# ============================================================
# ENHANCED 6-FEATURE EXTRACTION
# ============================================================
def extract_6_features(ecg_segment):
    ecg = np.asarray(ecg_segment).flatten()
    try:
        out = hamilton_segmenter(signal=ecg, sampling_rate=FS)
        r_peaks = np.asarray(out[0], dtype=np.int64)
        if len(r_peaks) < 4:
            return None
        
        rr_ms = np.diff(r_peaks) / FS * 1000.0
        rp_amp = ecg[r_peaks[1:]]
        rr_times = r_peaks[1:] / FS
        
        # QRS amplitude std (d=0.489 for 0.8-10 Hz)
        qrs_amps = []
        for i in range(len(r_peaks)):
            start = max(0, r_peaks[i] - int(0.08 * FS))
            end = min(len(ecg), r_peaks[i] + int(0.08 * FS))
            qrs_segment = ecg[start:end]
            qrs_amp = np.max(qrs_segment) - np.min(qrs_segment)
            qrs_amps.append(qrs_amp)
        qrs_amp_std = np.std(qrs_amps)
        
        # Mean HR (d=0.470)
        mean_rr = np.mean(rr_ms)
        mean_hr = 60000 / mean_rr if mean_rr > 0 else 0
        
        # Triangular Index (d=0.295)
        hist, _ = np.histogram(rr_ms, bins=50)
        tri_index = len(rr_ms) / np.max(hist) if np.max(hist) > 0 else 0
        
        # pNN50 (d=0.043 - weak but included)
        nn50 = np.sum(np.abs(np.diff(rr_ms)) > 50)
        pnn50 = (nn50 / len(rr_ms)) * 100 if len(rr_ms) > 0 else 0
        
        # Interpolate
        t_uni = np.arange(rr_times[0], rr_times[-1], 1/INTERP_FS)
        
        rr_uni = interp1d(rr_times, rr_ms, kind='cubic', 
                         fill_value='extrapolate')(t_uni)
        rp_uni = interp1d(rr_times, rp_amp, kind='cubic',
                         fill_value='extrapolate')(t_uni)
        
        qrs_std_uni = np.full_like(rr_uni, qrs_amp_std)
        mean_hr_uni = np.full_like(rr_uni, mean_hr)
        tri_idx_uni = np.full_like(rr_uni, tri_index)
        pnn50_uni = np.full_like(rr_uni, pnn50)
        
        n = len(rr_uni)
        def pad_or_crop(arr):
            if n >= SEG_LEN:
                s = (n - SEG_LEN) // 2
                return arr[s:s+SEG_LEN]
            else:
                pad = SEG_LEN - n
                return np.pad(arr, (pad//2, pad-pad//2), 'edge')
        
        features = np.stack([
            pad_or_crop(rr_uni),
            pad_or_crop(rp_uni),
            pad_or_crop(qrs_std_uni),
            pad_or_crop(mean_hr_uni),
            pad_or_crop(tri_idx_uni),
            pad_or_crop(pnn50_uni)
        ], axis=0).astype(np.float32)
        
        return features
        
    except:
        return None

print("\nExtracting 6 features...")
feature_list, y_list, subj_list = [], [], []

for i in range(len(X_raw)):
    feat = extract_6_features(X_raw[i])
    if feat is not None:
        feature_list.append(feat)
        y_list.append(y[i])
        subj_list.append(subj[i])
    
    if (i+1) % 5000 == 0:
        print(f"  Processed {i+1}/{len(X_raw)}...")

X_feat = np.array(feature_list)
y_valid = np.array(y_list)
subj_valid = np.array(subj_list)

print(f"Feature shape: {X_feat.shape}")

print("Applying subject-wise scaling...")
X_final = apply_subject_wise_scaling(X_feat, subj_valid)

print(f"Final shape: {X_final.shape}")
print(f"Unique subjects: {len(np.unique(subj_valid))}")

# ============================================================
# MODEL ARCHITECTURE
# ============================================================
class SEBlock(nn.Module):
    def __init__(self, channel, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channel, channel//reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channel//reduction, channel, bias=False),
            nn.Sigmoid())
    def forward(self, x):
        b, c, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1)
        return x * y.expand_as(x)

class MultiScaleBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        ch = out_ch // 4
        self.conv1_list = nn.ModuleList([
            nn.Conv1d(in_ch, ch, k, stride, k//2, bias=False)
            for k in [3,5,7,9]])
        self.bn1 = nn.BatchNorm1d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.conv2_list = nn.ModuleList([
            nn.Conv1d(out_ch, ch, k, 1, k//2, bias=False)
            for k in [3,5,7,9]])
        self.bn2 = nn.BatchNorm1d(out_ch)
        self.se = SEBlock(out_ch)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_ch, out_ch, 1, stride, bias=False),
                nn.BatchNorm1d(out_ch))
    def forward(self, x):
        identity = self.shortcut(x)
        out = torch.cat([c(x) for c in self.conv1_list], dim=1)
        out = self.relu(self.bn1(out))
        out = torch.cat([c(out) for c in self.conv2_list], dim=1)
        out = self.bn2(out)
        out = self.se(out)
        out += identity
        return self.relu(out)

class MultiScaleSEResNet(nn.Module):
    def __init__(self, in_channels=6):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(in_channels, 64, 7, 3, 3, bias=False),
            nn.BatchNorm1d(64), nn.ReLU(True),
            nn.MaxPool1d(3, 2, 1))
        self.layer1 = self._make_layer(64, 64, 2, 1)
        self.layer2 = self._make_layer(64, 128, 2, 2)
        self.layer3 = self._make_layer(128, 256, 2, 2)
        self.layer4 = self._make_layer(256, 512, 2, 2)
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(512, 1)
        self.drop = nn.Dropout(0.5)

    def _make_layer(self, in_c, out_c, blocks, stride):
        layers = [MultiScaleBlock(in_c, out_c, stride)]
        for _ in range(1, blocks):
            layers.append(MultiScaleBlock(out_c, out_c, 1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x); x = self.layer2(x)
        x = self.layer3(x); x = self.layer4(x)
        x = self.gap(x).squeeze(-1)
        return torch.sigmoid(self.fc(self.drop(x)))

# ============================================================
# STRATIFIED GROUP K-FOLD
# ============================================================
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
fold_results = []

print("\n" + "="*70)
print("StratifiedGroupKFold Cross-Validation (6 Features, 0.8-10 Hz)")
print("="*70)

for fold_idx, (train_idx, val_idx) in enumerate(
    sgkf.split(X_final, y_valid, groups=subj_valid)
):
    print(f"\n{'='*70}")
    print(f"FOLD {fold_idx+1}/{N_FOLDS}")
    print(f"{'='*70}")
    
    X_train = X_final[train_idx]
    y_train = y_valid[train_idx]
    X_val = X_final[val_idx]
    y_val = y_valid[val_idx]
    
    train_subjects = np.unique(subj_valid[train_idx])
    val_subjects = np.unique(subj_valid[val_idx])
    
    print(f"Train: {len(train_subjects)}subj, N={np.sum(y_train==0)}, "
          f"AH={np.sum(y_train==1)}")
    print(f"Val:   {len(val_subjects)}subj, N={np.sum(y_val==0)}, "
          f"AH={np.sum(y_val==1)}")
    
    cls_count = np.array([len(np.where(y_train==t)[0])
                         for t in np.unique(y_train)])
    weight = 1. / cls_count
    samp_weight = np.array([weight[int(t)] for t in y_train])
    sampler = WeightedRandomSampler(torch.from_numpy(samp_weight),
                                   len(samp_weight))
    
    train_loader = DataLoader(
        TensorDataset(torch.Tensor(X_train), torch.Tensor(y_train)),
        batch_size=BATCH_SIZE, sampler=sampler)
    
    val_loader = DataLoader(
        TensorDataset(torch.Tensor(X_val), torch.Tensor(y_val)),
        batch_size=BATCH_SIZE)
    
    model = MultiScaleSEResNet(in_channels=6).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 'min', factor=0.5, patience=3)
    criterion = nn.BCELoss()
    
    best_auc = 0
    best_state = None
    patience_counter = 0
    
    for epoch in range(EPOCHS):
        model.train()
        t_loss = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            out = model(xb).view(-1)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            t_loss += loss.item()
        
        model.eval()
        y_prob_v = []
        with torch.no_grad():
            for xb, _ in val_loader:
                y_prob_v.extend(model(xb.to(DEVICE)).cpu().numpy())
        
        val_auc = roc_auc_score(y_val, y_prob_v)
        scheduler.step(t_loss / len(train_loader))
        
        if epoch % 10 == 0:
            print(f"  Epoch {epoch+1:03d} | Loss: {t_loss/len(train_loader):.4f} | "
                  f"AUC: {val_auc:.4f}")
        
        if val_auc > best_auc:
            best_auc = val_auc
            best_state = {k: v.cpu().clone()
                         for k,v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= 12:
                print(f"  Early stop at epoch {epoch+1}")
                break
    
    model.load_state_dict(best_state)
    model.eval()
    y_prob = []
    with torch.no_grad():
        for xb, _ in val_loader:
            y_prob.extend(model(xb.to(DEVICE)).cpu().numpy())
    
    y_prob = np.convolve(np.array(y_prob).flatten(), 
                        np.ones(3)/3, mode='same')
    
    fpr, tpr, thresholds = roc_curve(y_val, y_prob)
    optimal_t = thresholds[np.argmax(tpr - fpr)]
    y_pred = (y_prob >= optimal_t).astype(int)
    
    auc = roc_auc_score(y_val, y_prob)
    acc = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()
    sens = tp / (tp + fn)
    spec = tn / (tn + fp)
    
    print(f"\nFold {fold_idx+1} Results:")
    print(f"  AUC={auc:.4f} | Acc={acc:.4f} | F1={f1:.4f} | "
          f"Sens={sens:.4f} | Spec={spec:.4f}")
    
    fold_results.append({
        'fold': fold_idx + 1,
        'auc': auc, 'acc': acc, 'f1': f1,
        'sens': sens, 'spec': spec
    })

# ============================================================
# SUMMARY
# ============================================================
print("\n" + "="*70)
print("FINAL SUMMARY — StratifiedGroupKFold (6 Features, 0.8-10 Hz)")
print("="*70)

for m in ['auc', 'acc', 'f1', 'sens', 'spec']:
    vals = [r[m] for r in fold_results]
    print(f"Mean {m.upper():<5}: {np.mean(vals):.4f} ± {np.std(vals):.4f}")

print("\n" + "="*70)
print("COMPARISON:")
print("="*70)
print("Baseline (2 feat, 0.8-10 Hz): AUC = 0.8037 ± 0.0359")
print(f"Enhanced (6 feat, 0.8-10 Hz): AUC = {np.mean([r['auc'] for r in fold_results]):.4f} ± {np.std([r['auc'] for r in fold_results]):.4f}")
improvement = (np.mean([r['auc'] for r in fold_results]) - 0.8037) * 100
print(f"Improvement: {improvement:+.2f}%")

print("\n" + "="*70)
print("FULL COMPARISON TABLE:")
print("="*70)
print(f"0.8-10 Hz | 2 feat | SGKF: AUC = 0.8037 ± 0.0359")
print(f"0.8-10 Hz | 6 feat | SGKF: AUC = {np.mean([r['auc'] for r in fold_results]):.4f} ± {np.std([r['auc'] for r in fold_results]):.4f}")
print(f"8-50 Hz   | 2 feat | SGKF: AUC = 0.8143 ± 0.0263")
print(f"8-50 Hz   | 6 feat | SGKF: AUC = 0.8191 ± 0.0153")

with open(SAVE_PATH, 'wb') as f:
    pickle.dump(fold_results, f)
print(f"\n✓ Results saved → {SAVE_PATH}")

Device: cuda
Enhanced 6-Feature Model with StratifiedGroupKFold (0.8-10 Hz)
Testing if enhanced features help with old preprocessing

X shape: (20092, 12000, 1)
Normal: 11548, AH: 8544

Extracting 6 features...
  Processed 5000/20092...
  Processed 10000/20092...
  Processed 15000/20092...
  Processed 20000/20092...
Feature shape: (20090, 6, 240)
Applying subject-wise scaling...
Final shape: (20090, 6, 240)
Unique subjects: 50

StratifiedGroupKFold Cross-Validation (6 Features, 0.8-10 Hz)

FOLD 1/5
Train: 40subj, N=9803, AH=6342
Val:   10subj, N=1743, AH=2202
  Epoch 001 | Loss: 0.6457 | AUC: 0.7078
  Epoch 011 | Loss: 0.4144 | AUC: 0.7532
  Early stop at epoch 19

Fold 1 Results:
  AUC=0.7895 | Acc=0.6923 | F1=0.6840 | Sens=0.5967 | Spec=0.8130

FOLD 2/5
Train: 40subj, N=8752, AH=7254
Val:   10subj, N=2794, AH=1290
  Epoch 001 | Loss: 0.6208 | AUC: 0.6422
  Epoch 011 | Loss: 0.3917 | AUC: 0.7096
  Epoch 021 | Loss: 0.2023 | AUC: 0.6895
  Early stop at epoch 25

Fold 2 Results:
  AUC=0

In [14]:
import numpy as np
import pickle
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from scipy.interpolate import interp1d
from biosppy.signals.ecg import hamilton_segmenter
from sklearn.metrics import (
    roc_auc_score, accuracy_score, confusion_matrix, roc_curve, f1_score
)
from sklearn.model_selection import StratifiedGroupKFold
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CONFIGURATION
# ============================================================
FS = 200
INTERP_FS = 4
SEG_LEN = 240
N_FOLDS = 5
BATCH_SIZE = 32
EPOCHS = 60
LR = 1e-3
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_PATH = r'E:\1D CNN\X_y_new_8_50hz.pkl'
SAVE_PATH = r'E:\1D CNN\results_2feat_sgkf_8_50hz.pkl'

print(f"Device: {DEVICE}")
print("Baseline: 2-Feature Model with StratifiedGroupKFold (8-50 Hz)")
print("="*70)

# ============================================================
# SUBJECT-WISE NORMALIZATION
# ============================================================
def apply_subject_wise_scaling(X, subject_ids):
    X_scaled = np.zeros_like(X)
    unique_subs = np.unique(subject_ids)
    for sub in unique_subs:
        mask = (subject_ids == sub)
        sub_data = X[mask]
        mean = np.mean(sub_data, axis=(0,2), keepdims=True)
        std = np.std(sub_data, axis=(0,2), keepdims=True) + 1e-8
        X_scaled[mask] = (sub_data - mean) / std
    return X_scaled

# ============================================================
# LOAD DATA
# ============================================================
with open(DATA_PATH, 'rb') as f:
    data = pickle.load(f)

X_raw = data['X']
y = np.array(data['y'])
subj = np.array(data['subject_ids'])

print(f"X shape: {X_raw.shape}")
print(f"Normal: {np.sum(y==0)}, AH: {np.sum(y==1)}")

# ============================================================
# 2-FEATURE EXTRACTION (RR intervals + R-peak amplitude)
# ============================================================
def extract_2_features(ecg_segment):
    ecg = np.asarray(ecg_segment).flatten()
    try:
        out = hamilton_segmenter(signal=ecg, sampling_rate=FS)
        r_peaks = np.asarray(out[0], dtype=np.int64)
        
        if len(r_peaks) < 4:
            return None
        
        rr_ms = np.diff(r_peaks) / FS * 1000.0
        rp_amp = ecg[r_peaks[1:]]
        rr_times = r_peaks[1:] / FS
        
        t_uni = np.arange(rr_times[0], rr_times[-1], 1/INTERP_FS)
        
        rr_uni = interp1d(rr_times, rr_ms, kind='cubic', 
                         fill_value='extrapolate')(t_uni)
        rp_uni = interp1d(rr_times, rp_amp, kind='cubic',
                         fill_value='extrapolate')(t_uni)
        
        n = len(rr_uni)
        
        if n >= SEG_LEN:
            start = (n - SEG_LEN) // 2
            rr_out = rr_uni[start:start+SEG_LEN]
            rp_out = rp_uni[start:start+SEG_LEN]
        else:
            pad = SEG_LEN - n
            rr_out = np.pad(rr_uni, (pad//2, pad-pad//2), 'edge')
            rp_out = np.pad(rp_uni, (pad//2, pad-pad//2), 'edge')
        
        return np.stack([rr_out, rp_out], axis=0).astype(np.float32)
        
    except:
        return None

print("\nExtracting 2 features (RR + R-peak)...")
feature_list, y_list, subj_list = [], [], []

for i in range(len(X_raw)):
    feat = extract_2_features(X_raw[i])
    if feat is not None:
        feature_list.append(feat)
        y_list.append(y[i])
        subj_list.append(subj[i])
    
    if (i+1) % 5000 == 0:
        print(f"  Processed {i+1}/{len(X_raw)}...")

X_feat = np.array(feature_list)
y_valid = np.array(y_list)
subj_valid = np.array(subj_list)

print(f"Feature shape: {X_feat.shape}")

print("Applying subject-wise scaling...")
X_final = apply_subject_wise_scaling(X_feat, subj_valid)

print(f"Final shape: {X_final.shape}")
print(f"Unique subjects: {len(np.unique(subj_valid))}")

# ============================================================
# MODEL ARCHITECTURE (2 input channels)
# ============================================================
class SEBlock(nn.Module):
    def __init__(self, channel, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channel, channel//reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channel//reduction, channel, bias=False),
            nn.Sigmoid())
    def forward(self, x):
        b, c, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1)
        return x * y.expand_as(x)

class MultiScaleBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        ch = out_ch // 4
        self.conv1_list = nn.ModuleList([
            nn.Conv1d(in_ch, ch, k, stride, k//2, bias=False)
            for k in [3,5,7,9]])
        self.bn1 = nn.BatchNorm1d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.conv2_list = nn.ModuleList([
            nn.Conv1d(out_ch, ch, k, 1, k//2, bias=False)
            for k in [3,5,7,9]])
        self.bn2 = nn.BatchNorm1d(out_ch)
        self.se = SEBlock(out_ch)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_ch, out_ch, 1, stride, bias=False),
                nn.BatchNorm1d(out_ch))
    def forward(self, x):
        identity = self.shortcut(x)
        out = torch.cat([c(x) for c in self.conv1_list], dim=1)
        out = self.relu(self.bn1(out))
        out = torch.cat([c(out) for c in self.conv2_list], dim=1)
        out = self.bn2(out)
        out = self.se(out)
        out += identity
        return self.relu(out)

class MultiScaleSEResNet(nn.Module):
    def __init__(self, in_channels=2):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(in_channels, 64, 7, 3, 3, bias=False),
            nn.BatchNorm1d(64), nn.ReLU(True),
            nn.MaxPool1d(3, 2, 1))
        self.layer1 = self._make_layer(64, 64, 2, 1)
        self.layer2 = self._make_layer(64, 128, 2, 2)
        self.layer3 = self._make_layer(128, 256, 2, 2)
        self.layer4 = self._make_layer(256, 512, 2, 2)
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(512, 1)
        self.drop = nn.Dropout(0.5)

    def _make_layer(self, in_c, out_c, blocks, stride):
        layers = [MultiScaleBlock(in_c, out_c, stride)]
        for _ in range(1, blocks):
            layers.append(MultiScaleBlock(out_c, out_c, 1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x); x = self.layer2(x)
        x = self.layer3(x); x = self.layer4(x)
        x = self.gap(x).squeeze(-1)
        return torch.sigmoid(self.fc(self.drop(x)))

# ============================================================
# STRATIFIED GROUP K-FOLD
# ============================================================
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
fold_results = []

print("\n" + "="*70)
print("StratifiedGroupKFold Cross-Validation (2 Features - Baseline)")
print("="*70)

for fold_idx, (train_idx, val_idx) in enumerate(
    sgkf.split(X_final, y_valid, groups=subj_valid)
):
    print(f"\n{'='*70}")
    print(f"FOLD {fold_idx+1}/{N_FOLDS}")
    print(f"{'='*70}")
    
    X_train = X_final[train_idx]
    y_train = y_valid[train_idx]
    X_val = X_final[val_idx]
    y_val = y_valid[val_idx]
    
    train_subjects = np.unique(subj_valid[train_idx])
    val_subjects = np.unique(subj_valid[val_idx])
    
    print(f"Train: {len(train_subjects)}subj, N={np.sum(y_train==0)}, "
          f"AH={np.sum(y_train==1)}")
    print(f"Val:   {len(val_subjects)}subj, N={np.sum(y_val==0)}, "
          f"AH={np.sum(y_val==1)}")
    
    cls_count = np.array([len(np.where(y_train==t)[0])
                         for t in np.unique(y_train)])
    weight = 1. / cls_count
    samp_weight = np.array([weight[int(t)] for t in y_train])
    sampler = WeightedRandomSampler(torch.from_numpy(samp_weight),
                                   len(samp_weight))
    
    train_loader = DataLoader(
        TensorDataset(torch.Tensor(X_train), torch.Tensor(y_train)),
        batch_size=BATCH_SIZE, sampler=sampler)
    
    val_loader = DataLoader(
        TensorDataset(torch.Tensor(X_val), torch.Tensor(y_val)),
        batch_size=BATCH_SIZE)
    
    model = MultiScaleSEResNet(in_channels=2).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 'min', factor=0.5, patience=3)
    criterion = nn.BCELoss()
    
    best_auc = 0
    best_state = None
    patience_counter = 0
    
    for epoch in range(EPOCHS):
        model.train()
        t_loss = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            out = model(xb).view(-1)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            t_loss += loss.item()
        
        model.eval()
        y_prob_v = []
        with torch.no_grad():
            for xb, _ in val_loader:
                y_prob_v.extend(model(xb.to(DEVICE)).cpu().numpy())
        
        val_auc = roc_auc_score(y_val, y_prob_v)
        scheduler.step(t_loss / len(train_loader))
        
        if epoch % 10 == 0:
            print(f"  Epoch {epoch+1:03d} | Loss: {t_loss/len(train_loader):.4f} | "
                  f"AUC: {val_auc:.4f}")
        
        if val_auc > best_auc:
            best_auc = val_auc
            best_state = {k: v.cpu().clone()
                         for k,v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= 12:
                print(f"  Early stop at epoch {epoch+1}")
                break
    
    model.load_state_dict(best_state)
    model.eval()
    y_prob = []
    with torch.no_grad():
        for xb, _ in val_loader:
            y_prob.extend(model(xb.to(DEVICE)).cpu().numpy())
    
    y_prob = np.convolve(np.array(y_prob).flatten(), 
                        np.ones(3)/3, mode='same')
    
    fpr, tpr, thresholds = roc_curve(y_val, y_prob)
    optimal_t = thresholds[np.argmax(tpr - fpr)]
    y_pred = (y_prob >= optimal_t).astype(int)
    
    auc = roc_auc_score(y_val, y_prob)
    acc = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()
    sens = tp / (tp + fn)
    spec = tn / (tn + fp)
    
    print(f"\nFold {fold_idx+1} Results:")
    print(f"  AUC={auc:.4f} | Acc={acc:.4f} | F1={f1:.4f} | "
          f"Sens={sens:.4f} | Spec={spec:.4f}")
    
    fold_results.append({
        'fold': fold_idx + 1,
        'auc': auc, 'acc': acc, 'f1': f1,
        'sens': sens, 'spec': spec
    })

# ============================================================
# SUMMARY
# ============================================================
print("\n" + "="*70)
print("FINAL SUMMARY — Baseline 2-Feature (8-50 Hz)")
print("="*70)

for m in ['auc', 'acc', 'f1', 'sens', 'spec']:
    vals = [r[m] for r in fold_results]
    print(f"Mean {m.upper():<5}: {np.mean(vals):.4f} ± {np.std(vals):.4f}")

print("\n" + "="*70)
print("Per-Fold Results:")
print("="*70)
for r in fold_results:
    print(f"Fold {r['fold']}: AUC={r['auc']:.4f} | Acc={r['acc']:.4f} | "
          f"F1={r['f1']:.4f} | Sens={r['sens']:.4f} | Spec={r['spec']:.4f}")

with open(SAVE_PATH, 'wb') as f:
    pickle.dump(fold_results, f)
print(f"\n✓ Results saved → {SAVE_PATH}")

Device: cuda
Baseline: 2-Feature Model with StratifiedGroupKFold (8-50 Hz)
X shape: (20092, 12000, 1)
Normal: 11548, AH: 8544

Extracting 2 features (RR + R-peak)...
  Processed 5000/20092...
  Processed 10000/20092...
  Processed 15000/20092...
  Processed 20000/20092...
Feature shape: (20091, 2, 240)
Applying subject-wise scaling...
Final shape: (20091, 2, 240)
Unique subjects: 50

StratifiedGroupKFold Cross-Validation (2 Features - Baseline)

FOLD 1/5
Train: 40subj, N=8752, AH=7253
Val:   10subj, N=2796, AH=1290
  Epoch 001 | Loss: 0.5703 | AUC: 0.7612
  Epoch 011 | Loss: 0.3269 | AUC: 0.7527
  Early stop at epoch 14

Fold 1 Results:
  AUC=0.8148 | Acc=0.7506 | F1=0.6388 | Sens=0.6984 | Spec=0.7747

FOLD 2/5
Train: 40subj, N=9493, AH=6635
Val:   10subj, N=2055, AH=1908
  Epoch 001 | Loss: 0.5886 | AUC: 0.7934
  Epoch 011 | Loss: 0.3648 | AUC: 0.8014
  Early stop at epoch 15

Fold 2 Results:
  AUC=0.8481 | Acc=0.7573 | F1=0.7476 | Sens=0.7469 | Spec=0.7669

FOLD 3/5
Train: 40subj, N=

In [ ]:
import numpy as np
import pickle
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from scipy.interpolate import interp1d
from biosppy.signals.ecg import hamilton_segmenter
from sklearn.metrics import (
    roc_auc_score, accuracy_score, confusion_matrix, roc_curve, f1_score
)
from sklearn.model_selection import StratifiedGroupKFold
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CONFIGURATION
# ============================================================
FS         = 200
INTERP_FS  = 4
SEG_LEN    = 240
N_FOLDS    = 5
BATCH_SIZE = 32
EPOCHS     = 60
LR         = 1e-3
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_PATH  = r'E:\1D CNN\X_y_new.pkl'  # 0.8-10 Hz data
SAVE_PATH  = r'E:\1D CNN\results_6feat_sgkf_0810hz.pkl'

print(f"Device: {DEVICE}")
print("Enhanced 6-Feature Model with StratifiedGroupKFold (0.8-10 Hz)")
print("="*70)
print("Testing if enhanced features help with old preprocessing")
print("="*70)

# ============================================================
# LOAD DATA
# ============================================================
with open(DATA_PATH, 'rb') as f:
    data = pickle.load(f)

X_raw = data['X']
y = np.array(data['y'])
subj = np.array(data['subject_ids'])

print(f"\nX shape: {X_raw.shape}")
print(f"Normal: {np.sum(y==0)}, AH: {np.sum(y==1)}")

# ============================================================
# SUBJECT-WISE NORMALIZATION
# ============================================================
def apply_subject_wise_scaling(X, subject_ids):
    X_scaled = np.zeros_like(X)
    unique_subs = np.unique(subject_ids)
    for sub in unique_subs:
        mask = (subject_ids == sub)
        sub_data = X[mask]
        mean = np.mean(sub_data, axis=(0,2), keepdims=True)
        std = np.std(sub_data, axis=(0,2), keepdims=True) + 1e-8
        X_scaled[mask] = (sub_data - mean) / std
    return X_scaled

# ============================================================
# ENHANCED 6-FEATURE EXTRACTION
# ============================================================
def extract_6_features(ecg_segment):
    ecg = np.asarray(ecg_segment).flatten()
    try:
        out = hamilton_segmenter(signal=ecg, sampling_rate=FS)
        r_peaks = np.asarray(out[0], dtype=np.int64)
        if len(r_peaks) < 4:
            return None
        
        rr_ms = np.diff(r_peaks) / FS * 1000.0
        rp_amp = ecg[r_peaks[1:]]
        rr_times = r_peaks[1:] / FS
        
        # QRS amplitude std (d=0.489 for 0.8-10 Hz)
        qrs_amps = []
        for i in range(len(r_peaks)):
            start = max(0, r_peaks[i] - int(0.08 * FS))
            end = min(len(ecg), r_peaks[i] + int(0.08 * FS))
            qrs_segment = ecg[start:end]
            qrs_amp = np.max(qrs_segment) - np.min(qrs_segment)
            qrs_amps.append(qrs_amp)
        qrs_amp_std = np.std(qrs_amps)
        
        # Mean HR (d=0.470)
        mean_rr = np.mean(rr_ms)
        mean_hr = 60000 / mean_rr if mean_rr > 0 else 0
        
        # Triangular Index (d=0.295)
        hist, _ = np.histogram(rr_ms, bins=50)
        tri_index = len(rr_ms) / np.max(hist) if np.max(hist) > 0 else 0
        
        # pNN50 (d=0.043 - weak but included)
        nn50 = np.sum(np.abs(np.diff(rr_ms)) > 50)
        pnn50 = (nn50 / len(rr_ms)) * 100 if len(rr_ms) > 0 else 0
        
        # Interpolate
        t_uni = np.arange(rr_times[0], rr_times[-1], 1/INTERP_FS)
        
        rr_uni = interp1d(rr_times, rr_ms, kind='cubic', 
                         fill_value='extrapolate')(t_uni)
        rp_uni = interp1d(rr_times, rp_amp, kind='cubic',
                         fill_value='extrapolate')(t_uni)
        
        qrs_std_uni = np.full_like(rr_uni, qrs_amp_std)
        mean_hr_uni = np.full_like(rr_uni, mean_hr)
        tri_idx_uni = np.full_like(rr_uni, tri_index)
        pnn50_uni = np.full_like(rr_uni, pnn50)
        
        n = len(rr_uni)
        def pad_or_crop(arr):
            if n >= SEG_LEN:
                s = (n - SEG_LEN) // 2
                return arr[s:s+SEG_LEN]
            else:
                pad = SEG_LEN - n
                return np.pad(arr, (pad//2, pad-pad//2), 'edge')
        
        features = np.stack([
            pad_or_crop(rr_uni),
            pad_or_crop(rp_uni),
            pad_or_crop(qrs_std_uni),
            pad_or_crop(mean_hr_uni),
            pad_or_crop(tri_idx_uni),
            pad_or_crop(pnn50_uni)
        ], axis=0).astype(np.float32)
        
        return features
        
    except:
        return None

print("\nExtracting 6 features...")
feature_list, y_list, subj_list = [], [], []

for i in range(len(X_raw)):
    feat = extract_6_features(X_raw[i])
    if feat is not None:
        feature_list.append(feat)
        y_list.append(y[i])
        subj_list.append(subj[i])
    
    if (i+1) % 5000 == 0:
        print(f"  Processed {i+1}/{len(X_raw)}...")

X_feat = np.array(feature_list)
y_valid = np.array(y_list)
subj_valid = np.array(subj_list)

print(f"Feature shape: {X_feat.shape}")

print("Applying subject-wise scaling...")
X_final = apply_subject_wise_scaling(X_feat, subj_valid)

print(f"Final shape: {X_final.shape}")
print(f"Unique subjects: {len(np.unique(subj_valid))}")

# ============================================================
# MODEL ARCHITECTURE
# ============================================================
class SEBlock(nn.Module):
    def __init__(self, channel, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channel, channel//reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channel//reduction, channel, bias=False),
            nn.Sigmoid())
    def forward(self, x):
        b, c, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1)
        return x * y.expand_as(x)

class MultiScaleBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        ch = out_ch // 4
        self.conv1_list = nn.ModuleList([
            nn.Conv1d(in_ch, ch, k, stride, k//2, bias=False)
            for k in [3,5,7,9]])
        self.bn1 = nn.BatchNorm1d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.conv2_list = nn.ModuleList([
            nn.Conv1d(out_ch, ch, k, 1, k//2, bias=False)
            for k in [3,5,7,9]])
        self.bn2 = nn.BatchNorm1d(out_ch)
        self.se = SEBlock(out_ch)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_ch, out_ch, 1, stride, bias=False),
                nn.BatchNorm1d(out_ch))
    def forward(self, x):
        identity = self.shortcut(x)
        out = torch.cat([c(x) for c in self.conv1_list], dim=1)
        out = self.relu(self.bn1(out))
        out = torch.cat([c(out) for c in self.conv2_list], dim=1)
        out = self.bn2(out)
        out = self.se(out)
        out += identity
        return self.relu(out)

class MultiScaleSEResNet(nn.Module):
    def __init__(self, in_channels=6):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(in_channels, 64, 7, 3, 3, bias=False),
            nn.BatchNorm1d(64), nn.ReLU(True),
            nn.MaxPool1d(3, 2, 1))
        self.layer1 = self._make_layer(64, 64, 2, 1)
        self.layer2 = self._make_layer(64, 128, 2, 2)
        self.layer3 = self._make_layer(128, 256, 2, 2)
        self.layer4 = self._make_layer(256, 512, 2, 2)
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(512, 1)
        self.drop = nn.Dropout(0.5)

    def _make_layer(self, in_c, out_c, blocks, stride):
        layers = [MultiScaleBlock(in_c, out_c, stride)]
        for _ in range(1, blocks):
            layers.append(MultiScaleBlock(out_c, out_c, 1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x); x = self.layer2(x)
        x = self.layer3(x); x = self.layer4(x)
        x = self.gap(x).squeeze(-1)
        return torch.sigmoid(self.fc(self.drop(x)))

# ============================================================
# STRATIFIED GROUP K-FOLD
# ============================================================
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
fold_results = []

print("\n" + "="*70)
print("StratifiedGroupKFold Cross-Validation (6 Features, 0.8-10 Hz)")
print("="*70)

for fold_idx, (train_idx, val_idx) in enumerate(
    sgkf.split(X_final, y_valid, groups=subj_valid)
):
    print(f"\n{'='*70}")
    print(f"FOLD {fold_idx+1}/{N_FOLDS}")
    print(f"{'='*70}")
    
    X_train = X_final[train_idx]
    y_train = y_valid[train_idx]
    X_val = X_final[val_idx]
    y_val = y_valid[val_idx]
    
    train_subjects = np.unique(subj_valid[train_idx])
    val_subjects = np.unique(subj_valid[val_idx])
    
    print(f"Train: {len(train_subjects)}subj, N={np.sum(y_train==0)}, "
          f"AH={np.sum(y_train==1)}")
    print(f"Val:   {len(val_subjects)}subj, N={np.sum(y_val==0)}, "
          f"AH={np.sum(y_val==1)}")
    
    cls_count = np.array([len(np.where(y_train==t)[0])
                         for t in np.unique(y_train)])
    weight = 1. / cls_count
    samp_weight = np.array([weight[int(t)] for t in y_train])
    sampler = WeightedRandomSampler(torch.from_numpy(samp_weight),
                                   len(samp_weight))
    
    train_loader = DataLoader(
        TensorDataset(torch.Tensor(X_train), torch.Tensor(y_train)),
        batch_size=BATCH_SIZE, sampler=sampler)
    
    val_loader = DataLoader(
        TensorDataset(torch.Tensor(X_val), torch.Tensor(y_val)),
        batch_size=BATCH_SIZE)
    
    model = MultiScaleSEResNet(in_channels=6).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 'min', factor=0.5, patience=3)
    criterion = nn.BCELoss()
    
    best_auc = 0
    best_state = None
    patience_counter = 0
    
    for epoch in range(EPOCHS):
        model.train()
        t_loss = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            out = model(xb).view(-1)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            t_loss += loss.item()
        
        model.eval()
        y_prob_v = []
        with torch.no_grad():
            for xb, _ in val_loader:
                y_prob_v.extend(model(xb.to(DEVICE)).cpu().numpy())
        
        val_auc = roc_auc_score(y_val, y_prob_v)
        scheduler.step(t_loss / len(train_loader))
        
        if epoch % 10 == 0:
            print(f"  Epoch {epoch+1:03d} | Loss: {t_loss/len(train_loader):.4f} | "
                  f"AUC: {val_auc:.4f}")
        
        if val_auc > best_auc:
            best_auc = val_auc
            best_state = {k: v.cpu().clone()
                         for k,v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= 12:
                print(f"  Early stop at epoch {epoch+1}")
                break
    
    model.load_state_dict(best_state)
    model.eval()
    y_prob = []
    with torch.no_grad():
        for xb, _ in val_loader:
            y_prob.extend(model(xb.to(DEVICE)).cpu().numpy())
    
    y_prob = np.convolve(np.array(y_prob).flatten(), 
                        np.ones(3)/3, mode='same')
    
    fpr, tpr, thresholds = roc_curve(y_val, y_prob)
    optimal_t = thresholds[np.argmax(tpr - fpr)]
    y_pred = (y_prob >= optimal_t).astype(int)
    
    auc = roc_auc_score(y_val, y_prob)
    acc = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()
    sens = tp / (tp + fn)
    spec = tn / (tn + fp)
    
    print(f"\nFold {fold_idx+1} Results:")
    print(f"  AUC={auc:.4f} | Acc={acc:.4f} | F1={f1:.4f} | "
          f"Sens={sens:.4f} | Spec={spec:.4f}")
    
    fold_results.append({
        'fold': fold_idx + 1,
        'auc': auc, 'acc': acc, 'f1': f1,
        'sens': sens, 'spec': spec
    })

# ============================================================
# SUMMARY
# ============================================================
print("\n" + "="*70)
print("FINAL SUMMARY — StratifiedGroupKFold (6 Features, 0.8-10 Hz)")
print("="*70)

for m in ['auc', 'acc', 'f1', 'sens', 'spec']:
    vals = [r[m] for r in fold_results]
    print(f"Mean {m.upper():<5}: {np.mean(vals):.4f} ± {np.std(vals):.4f}")

print("\n" + "="*70)
print("COMPARISON:")
print("="*70)
print("Baseline (2 feat, 0.8-10 Hz): AUC = 0.8037 ± 0.0359")
print(f"Enhanced (6 feat, 0.8-10 Hz): AUC = {np.mean([r['auc'] for r in fold_results]):.4f} ± {np.std([r['auc'] for r in fold_results]):.4f}")
improvement = (np.mean([r['auc'] for r in fold_results]) - 0.8037) * 100
print(f"Improvement: {improvement:+.2f}%")

print("\n" + "="*70)
print("FULL COMPARISON TABLE:")
print("="*70)
print(f"0.8-10 Hz | 2 feat | SGKF: AUC = 0.8037 ± 0.0359")
print(f"0.8-10 Hz | 6 feat | SGKF: AUC = {np.mean([r['auc'] for r in fold_results]):.4f} ± {np.std([r['auc'] for r in fold_results]):.4f}")
print(f"8-50 Hz   | 2 feat | SGKF: AUC = 0.8143 ± 0.0263")
print(f"8-50 Hz   | 6 feat | SGKF: AUC = 0.8191 ± 0.0153")

with open(SAVE_PATH, 'wb') as f:
    pickle.dump(fold_results, f)
print(f"\n✓ Results saved → {SAVE_PATH}")